# ROGII Ridge Artifact Parameter Experiments


## Final Prediction Controls

Set `SUBMISSION_PROFILE` in the first code cell. Use `ridge_artifact_experiment` for the sp45 projection parameter set or nearby probes, `ridge_artifact_exact` for the older ridge-artifact preset, or `fast_pf_selector` for the direct PF/beam selector baseline.

The ridge artifact path first combines a saved Ridge estimate with a physical/PF heuristic:

$$
T_i^{\mathrm{blend}} = w_r T_i^{\mathrm{ridge}} + (1-w_r)T_i^{\mathrm{heur}}.
$$

The heuristic estimate is controlled by the PF ensemble size and initialization spread:

$$
T_i^{\mathrm{heur}} = H_i(N_p, S, \sigma_0),
$$

where $N_p$ is the number of particles, $S$ is the number of PF seeds, and $\sigma_0$ is the initial TVT spread around the last known anchor.

An optional per-well projection then smooths the trajectory in the coordinate $U=T+Z$:

$$
U_i^{\mathrm{blend}} = T_i^{\mathrm{blend}} + Z_i - A_w,
$$

$$
U_i^{\mathrm{proj}} = P_d(s_i), \qquad s_i = \frac{MD_i-MD_w^{\mathrm{last}}}{MD_w^{\mathrm{end}}-MD_w^{\mathrm{last}}},
$$

$$
T_i^{\mathrm{final}} = A_w + U_i^{\mathrm{proj}} - Z_i.
$$

This projection is controlled by the degree $d$ and a robust fit loop.

The fast selector path submits the target-free PF/beam selector directly:

$$
T_i^{\mathrm{final}} = T_i^{\mathrm{selector}}.
$$

`PF_SELECTOR_USE_SAME_WELL_PHYSICAL=True` allows the same-well physical/contact shortcut when a test well also appears in train. Setting it to `False` forces the PF/beam selector path.


In [1]:
# Submission profile.
# Choices:
# - ridge_artifact_experiment: use the parameter values below.
# - ridge_artifact_exact: force the older reference preset, w_r=0.30, N_p=600, S=150, sigma_0=2.0.
# - fast_pf_selector: direct target-free PF/beam selector baseline.
SUBMISSION_PROFILE = 'ridge_artifact_experiment'

# Ridge artifact experiment parameters.
RIDGE_ARTIFACT_EXPERIMENT_LABEL = "ridge_exp_w030_p500_s128_sp45_proj_d4"
RIDGE_ARTIFACT_RIDGE_WEIGHT = 0.30
RIDGE_ARTIFACT_PF_N_PARTICLES = 500
RIDGE_ARTIFACT_PF_N_SEEDS = 128
RIDGE_ARTIFACT_PF_INIT_SPREAD = 4.5

# Optional per-well projection in U = TVT + Z - anchor space.
RIDGE_ARTIFACT_APPLY_PROJECTION = True
RIDGE_ARTIFACT_PROJECTION_DEGREE = 4
RIDGE_ARTIFACT_PROJECTION_ROBUST_ITERS = 4
RIDGE_ARTIFACT_PROJECTION_ROBUST_C = 2.0

# Useful nearby probes:
# - sp45-style: RIDGE_ARTIFACT_PF_INIT_SPREAD=4.5, RIDGE_ARTIFACT_PF_N_PARTICLES=500, RIDGE_ARTIFACT_PF_N_SEEDS=128
# - mild-spread: RIDGE_ARTIFACT_PF_INIT_SPREAD=4.25, RIDGE_ARTIFACT_PF_N_PARTICLES=500, RIDGE_ARTIFACT_PF_N_SEEDS=128
# - projection probe: keep ridge/PF fixed and vary RIDGE_ARTIFACT_PROJECTION_DEGREE in {3, 4, 5, 6}.

# Target-free PF/beam selector settings.
PF_SELECTOR_N_PARTICLES = 500
PF_SELECTOR_N_SEEDS = 64
PF_SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)
PF_SELECTOR_AS_AUX_GATED_MAX_WEIGHT = 0.015
PF_SELECTOR_AS_AUX_GATED_SCALE = 4.0
# True enables the overlap-aware selector shortcut; False disables the same-well physical shortcut.
PF_SELECTOR_USE_SAME_WELL_PHYSICAL = True

# Artifact-backed ridge dataset roots.
RIDGE_ARTIFACT_ROOTS = [
    '/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts',
    '/kaggle/input/wellbore-geology-prediction-artifacts',
]

# Data roots.
COMPETITION_DATA_ROOTS = [
    '/kaggle/input/rogii-wellbore-geology-prediction',
    '/kaggle/input/competitions/rogii-wellbore-geology-prediction',
]

# Optional TVT clipping. Keep None unless calibrated bounds are known.
TVT_CLIP_MIN = None
TVT_CLIP_MAX = None


In [2]:
import re

_profile = str(SUBMISSION_PROFILE).strip().lower()
_valid_profiles = {
    'ridge_artifact_experiment',
    'ridge_artifact_exact',
    'fast_pf_selector',
}
if _profile not in _valid_profiles:
    raise ValueError(f'SUBMISSION_PROFILE must be one of {sorted(_valid_profiles)}')

RUN_FAST_PF_SELECTOR_ONLY = _profile == 'fast_pf_selector'
RUN_FAST_PF_SELECTOR_128 = False
RUN_RIDGE_ARTIFACT_EXPERIMENT = _profile == 'ridge_artifact_experiment'
RUN_RIDGE_ARTIFACT_EXACT = _profile == 'ridge_artifact_exact'
RUN_RIDGE_ARTIFACT_PROFILE = _profile in {'ridge_artifact_experiment', 'ridge_artifact_exact'}
RUN_TARGET_FREE_SELECTOR_CANDIDATE = _profile == 'fast_pf_selector'

if RUN_RIDGE_ARTIFACT_EXACT:
    RIDGE_ARTIFACT_RIDGE_WEIGHT = 0.30
    RIDGE_ARTIFACT_PF_N_PARTICLES = 600
    RIDGE_ARTIFACT_PF_N_SEEDS = 150
    RIDGE_ARTIFACT_PF_INIT_SPREAD = 2.0
    RIDGE_ARTIFACT_APPLY_PROJECTION = False
    RIDGE_ARTIFACT_PROFILE_LABEL = 'ridge_artifact_exact'
    FINAL_V7_CANDIDATE = 'ridge_artifact_exact'
elif RUN_RIDGE_ARTIFACT_EXPERIMENT:
    RIDGE_ARTIFACT_RIDGE_WEIGHT = float(RIDGE_ARTIFACT_RIDGE_WEIGHT)
    RIDGE_ARTIFACT_PF_N_PARTICLES = int(RIDGE_ARTIFACT_PF_N_PARTICLES)
    RIDGE_ARTIFACT_PF_N_SEEDS = int(RIDGE_ARTIFACT_PF_N_SEEDS)
    RIDGE_ARTIFACT_PF_INIT_SPREAD = float(RIDGE_ARTIFACT_PF_INIT_SPREAD)
    _label = str(globals().get('RIDGE_ARTIFACT_EXPERIMENT_LABEL', 'ridge_artifact_experiment')).strip()
    _label = re.sub(r'[^A-Za-z0-9_.-]+', '_', _label).strip('_') or 'ridge_artifact_experiment'
    RIDGE_ARTIFACT_PROFILE_LABEL = _label
    FINAL_V7_CANDIDATE = _label
else:
    RIDGE_ARTIFACT_PROFILE_LABEL = None
    FINAL_V7_CANDIDATE = 'pf_selector'

if RUN_RIDGE_ARTIFACT_PROFILE:
    if not (0.0 <= float(RIDGE_ARTIFACT_RIDGE_WEIGHT) <= 1.0):
        raise ValueError('RIDGE_ARTIFACT_RIDGE_WEIGHT must be in [0, 1].')
    if int(RIDGE_ARTIFACT_PF_N_PARTICLES) <= 0:
        raise ValueError('RIDGE_ARTIFACT_PF_N_PARTICLES must be positive.')
    if int(RIDGE_ARTIFACT_PF_N_SEEDS) <= 0:
        raise ValueError('RIDGE_ARTIFACT_PF_N_SEEDS must be positive.')
    if float(RIDGE_ARTIFACT_PF_INIT_SPREAD) < 0:
        raise ValueError('RIDGE_ARTIFACT_PF_INIT_SPREAD must be non-negative.')
    RIDGE_ARTIFACT_APPLY_PROJECTION = bool(globals().get('RIDGE_ARTIFACT_APPLY_PROJECTION', False))
    RIDGE_ARTIFACT_PROJECTION_DEGREE = int(globals().get('RIDGE_ARTIFACT_PROJECTION_DEGREE', 5))
    RIDGE_ARTIFACT_PROJECTION_ROBUST_ITERS = int(globals().get('RIDGE_ARTIFACT_PROJECTION_ROBUST_ITERS', 4))
    RIDGE_ARTIFACT_PROJECTION_ROBUST_C = float(globals().get('RIDGE_ARTIFACT_PROJECTION_ROBUST_C', 2.0))
    if RIDGE_ARTIFACT_PROJECTION_DEGREE < 1:
        raise ValueError('RIDGE_ARTIFACT_PROJECTION_DEGREE must be positive.')
    if RIDGE_ARTIFACT_PROJECTION_ROBUST_ITERS < 0:
        raise ValueError('RIDGE_ARTIFACT_PROJECTION_ROBUST_ITERS must be non-negative.')
    if RIDGE_ARTIFACT_PROJECTION_ROBUST_C <= 0:
        raise ValueError('RIDGE_ARTIFACT_PROJECTION_ROBUST_C must be positive.')

profile_summary = {
    'submission_profile': SUBMISSION_PROFILE,
    'final_candidate': FINAL_V7_CANDIDATE,
    'active_engine': (
        'artifact ridge parameter experiment' if RUN_RIDGE_ARTIFACT_EXPERIMENT else
        'artifact ridge exact preset' if RUN_RIDGE_ARTIFACT_EXACT else
        'PF selector only'
    ),
    'ridge_weight_w_r': float(globals().get('RIDGE_ARTIFACT_RIDGE_WEIGHT', 0.0)) if RUN_RIDGE_ARTIFACT_PROFILE else None,
    'pf_particles_N_p': int(globals().get('RIDGE_ARTIFACT_PF_N_PARTICLES', 0)) if RUN_RIDGE_ARTIFACT_PROFILE else None,
    'pf_seeds_S': int(globals().get('RIDGE_ARTIFACT_PF_N_SEEDS', 0)) if RUN_RIDGE_ARTIFACT_PROFILE else None,
    'pf_init_spread_sigma_0': float(globals().get('RIDGE_ARTIFACT_PF_INIT_SPREAD', 0.0)) if RUN_RIDGE_ARTIFACT_PROFILE else None,
    'projection_enabled': bool(globals().get('RIDGE_ARTIFACT_APPLY_PROJECTION', False)) if RUN_RIDGE_ARTIFACT_PROFILE else None,
    'projection_degree_d': int(globals().get('RIDGE_ARTIFACT_PROJECTION_DEGREE', 0)) if RUN_RIDGE_ARTIFACT_PROFILE else None,
}

print(profile_summary)


{'submission_profile': 'ridge_artifact_experiment', 'final_candidate': 'ridge_exp_w030_p500_s128_sp45_proj_d4', 'active_engine': 'artifact ridge parameter experiment', 'ridge_weight_w_r': 0.3, 'pf_particles_N_p': 500, 'pf_seeds_S': 128, 'pf_init_spread_sigma_0': 4.5, 'projection_enabled': True, 'projection_degree_d': 4}


In [3]:
# Shared path normalization.
from pathlib import Path

COMPETITION_DATA_ROOTS = [Path(p) for p in COMPETITION_DATA_ROOTS]
FINAL_SIDECAR_SOURCE_LABEL = str(globals().get('FINAL_BASE_SOURCE_LABEL', 'base_only'))
FINAL_SIDECAR_AVAILABLE = False
FINAL_SIDECAR_AUTO_DISABLED_REASON = ''


In [4]:
# Configure imports, data locations, and output paths.
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:.5g}')

KAGGLE_DATA_DIRS = [
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
]
LOCAL_DATA_DIR = Path('.')
CANDIDATE_DATA_DIRS = KAGGLE_DATA_DIRS + [LOCAL_DATA_DIR]
DATA_DIR = next(
    (
        p
        for p in CANDIDATE_DATA_DIRS
        if (p / 'train').exists() and (p / 'sample_submission.csv').exists()
    ),
    LOCAL_DATA_DIR,
)
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
SAMPLE_SUBMISSION = DATA_DIR / 'sample_submission.csv'
KAGGLE_WORKING_DIR = Path('/kaggle/working')
KAGGLE_NOTEBOOK_RUN = KAGGLE_WORKING_DIR.exists()
OUTPUT_DIR = KAGGLE_WORKING_DIR if KAGGLE_NOTEBOOK_RUN else DATA_DIR
FINAL_SUBMISSION_OUTPUT = OUTPUT_DIR / 'submission.csv'

DATA_DIR_LABEL = './' if DATA_DIR == LOCAL_DATA_DIR else DATA_DIR.as_posix()
print('DATA_DIR:', DATA_DIR_LABEL)
print('train exists:', TRAIN_DIR.exists())
print('test exists:', TEST_DIR.exists())
print('sample_submission exists:', SAMPLE_SUBMISSION.exists())
print('OUTPUT_DIR:', OUTPUT_DIR.as_posix() if OUTPUT_DIR != DATA_DIR else DATA_DIR_LABEL)


DATA_DIR: /kaggle/input/competitions/rogii-wellbore-geology-prediction
train exists: True
test exists: True
sample_submission exists: True
OUTPUT_DIR: /kaggle/working


## Execution Engines

Only one engine writes `submission.csv` for a run. Ridge profiles skip the selector engine and write the ridge-artifact blend. `fast_pf_selector` skips the ridge engine and writes the direct PF/beam selector output.


In [5]:
# Super-stack final submission engine.
RUN_SUPER_STACK_SOLUTION = (
    bool(KAGGLE_NOTEBOOK_RUN)
    and not bool(globals().get('RUN_RIDGE_ARTIFACT_PROFILE', False))
)
SUPER_STACK_SUBMISSION_OUTPUT = FINAL_SUBMISSION_OUTPUT

if not RUN_SUPER_STACK_SOLUTION:
    if bool(globals().get('RUN_RIDGE_ARTIFACT_PROFILE', False)):
        print(f"Super-stack final solution is skipped for {globals().get('SUBMISSION_PROFILE', 'ridge_artifact_experiment')} profile.")
    else:
        print('Super-stack final solution is skipped outside Kaggle submission runs.')
else:
    # ─ Imports & Config ──────────────────────────────────────────────
    from pathlib import Path
    from scipy.interpolate import interp1d
    from scipy.spatial import cKDTree
    from scipy.signal import savgol_filter
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    from sklearn.metrics import root_mean_squared_error
    try:
        from numba import njit
    except Exception:
        def njit(*args, **kwargs):
            def _decorator(func):
                return func
            return _decorator
    if not bool(globals().get('RUN_FAST_PF_SELECTOR_ONLY', False)):
        from catboost import CatBoostRegressor, Pool
        import lightgbm as lgb
    else:
        CatBoostRegressor = Pool = None
        lgb = None
    from joblib import Parallel, delayed
    import numpy as np, pandas as pd
    import glob, gc, time, multiprocessing, warnings, json
    warnings.filterwarnings("ignore")

    SEED = 42; np.random.seed(SEED)
    NCPU = 1  # DTW/PF feature building is RAM-bound; serial builds are safer and deterministic.

    def stable_seed(wid, salt=SEED):
        return int((sum((i + 1) * ord(ch) for i, ch in enumerate(str(wid))) + int(salt)) % (2**32 - 1))

    def _gpu_names():
        import subprocess
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True, check=False).stdout.strip()
        except Exception:
            out = ""
        return [line.strip() for line in out.splitlines() if line.strip()]

    REFERENCE_GPU_NAMES = _gpu_names()
    if KAGGLE_NOTEBOOK_RUN and not REFERENCE_GPU_NAMES and not bool(globals().get('RUN_FAST_PF_SELECTOR_ONLY', False)):
        raise RuntimeError("Super-stack final solution requires a Kaggle GPU accelerator.")

    def _find():
        if 'DATA_DIR' in globals() and (Path(DATA_DIR) / "train").exists():
            return Path(DATA_DIR)
        for p in [Path("/kaggle/input/rogii-wellbore-geology-prediction"),
                  Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")]:
            if (p / "train").exists():
                return p
        kaggle_input = Path("/kaggle/input")
        if kaggle_input.exists():
            for p in kaggle_input.glob("*/sample_submission.csv"):
                return p.parent
        local = Path(".")
        if (local / "train").exists() and (local / "sample_submission.csv").exists():
            return local
        raise FileNotFoundError("Data not found")

    DATA      = _find()
    TRAIN_DIR = DATA / "train"
    TEST_DIR  = DATA / "test"
    SAMPLE    = DATA / "sample_submission.csv"
    SUPER_STACK_SUBMISSION_OUTPUT = FINAL_SUBMISSION_OUTPUT
    OUT       = FINAL_SUBMISSION_OUTPUT

    FORMATIONS = ["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
    PLANE_K    = 10          # centroid plane-fit neighbors
    DENSE_SPW  = 60          # dense samples per well (raised from 40)
    DENSE_K    = 20
    N_SPLITS   = 5

    # Beam configs: (beam_size, move_cost, emit_scale, smooth_r, tag)
    BEAMS = [
        (10, 20.0, 144.0, 2, "cons"),
        (10,  8.0,  64.0, 2, "loose"),
        ( 8, 35.0, 220.0, 1, "vcons"),
        (10, 14.0,  90.0, 5, "sm5"),
        (20,  4.0,  36.0, 3, "vloose"),
    ]

    # Particle filter (reduced particles for speed)
    PF_N  = 300;   ANCC_N = 300
    PF_MOM = 0.993; PF_VN  = 0.005; PF_PN  = 0.01
    PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
    PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
    PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
    ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
    ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001

    # Constrained / stochastic DTW. These are the main v7 additions.
    DTW_RADII = (20, 50, 100)
    DTW_STRIDE = 3
    DTW_STOCH_K = 6
    DTW_STOCH_TEMP = 3.0

    # Model params
    LGB_P = dict(boosting_type="gbdt",learning_rate=0.04,num_leaves=127,
                 min_child_samples=20,subsample=0.8,colsample_bytree=0.8,
                 reg_lambda=5.,reg_alpha=0.1,objective="regression",
                 verbose=-1,n_jobs=-1,
                 device_type="gpu",gpu_use_dp=False,max_bin=255)
    LGB_SEEDS = [42, 7, 123]

    CB_P = dict(iterations=5000,learning_rate=0.04,depth=8,l2_leaf_reg=3.,
                min_data_in_leaf=20,loss_function="RMSE",
                random_seed=42,task_type="GPU",devices=("0:1" if len(REFERENCE_GPU_NAMES) >= 2 else "0"),
                od_type="Iter",od_wait=150,verbose=0)

    print("GPUs:", " | ".join(REFERENCE_GPU_NAMES) if REFERENCE_GPU_NAMES else "none")
    print(f"CPUs={NCPU}  train={len(list(TRAIN_DIR.glob('*__horizontal_well.csv')))} wells")


    # ─ Helpers + Beam Search ─────────────────────────────────────────
    def nn_idx(arr, v):
        i=int(np.searchsorted(arr,v,'left'))
        if i>=len(arr): return len(arr)-1
        if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
        return i

    def robust_slope(x, y, w=None):
        x=np.asarray(x,float); y=np.asarray(y,float)
        m=np.isfinite(x)&np.isfinite(y)
        if m.sum()<2: return 0.
        if np.std(x[m])<1e-6: return 0.
        return float(np.polyfit(x[m],y[m],1)[0])

    def affine_cal(kgr, tw_at_k, min_pts=20):
        v=np.isfinite(kgr)&np.isfinite(tw_at_k)
        if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
            return 1., float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
        a,b=np.polyfit(tw_at_k[v],kgr[v],1)
        return float(a),float(b)

    def self_corr_tvt(kgr, ktvt, hgr, hw=15, stride=3):
        win=2*hw+1; nk=len(kgr); nh=len(hgr)
        if nk<win+1 or nh==0:
            return np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32)
        kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
        sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
        if M==0: return np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32)
        C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
        Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
        hp=np.pad(hg,hw,mode='edge')
        H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
        Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
        ncc=Hn@Cn.T/win
        best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
        ctrs=np.clip(sts[best]+hw,0,nk-1)
        return ktvt[ctrs].astype(np.float32),score

    def beam_search(gr_h, tw_tvt, tw_gr, start_tvt, bs=10, mc=20., es=144., r=2):
        tw_tvt=np.asarray(tw_tvt,np.float32); tw_gr=np.asarray(tw_gr,np.float32)
        T=len(tw_tvt); fb=float(np.nanmean(tw_gr))
        sg=pd.Series(gr_h,dtype='float32').interpolate(limit_direction='both').fillna(fb)
        if r>0: sg=sg.rolling(r*2+1,center=True,min_periods=1).mean()
        sg=sg.to_numpy(np.float32)
        si=nn_idx(tw_tvt,start_tvt)
        bi=np.full(bs,si,np.int32); bc=np.zeros(bs,np.float64)
        ns=len(sg); bps=np.empty((ns,bs),np.int32); bpb=np.empty((ns,bs),np.int32)
        for s,gv in enumerate(sg):
            ci=np.clip(bi[:,None]+np.array([-1,0,1]),0,T-1)
            em=(gv-tw_gr[ci])**2/es; mv=mc*np.array([1,0,1])[None,:]
            cc=bc[:,None]+em+mv
            fi=ci.ravel(); fc=cc.ravel(); fp=np.repeat(np.arange(bs),3)
            ord=np.argsort(fc, kind='stable'); kept=[]; seen=set()
            for o in ord:
                t=int(fi[o])
                if t not in seen: seen.add(t); kept.append(o)
                if len(kept)==bs: break
            while len(kept)<bs: kept.append(kept[-1])
            kept=np.array(kept,np.int32)
            bps[s]=fp[kept]; bpb[s]=fi[kept]
            bi=fi[kept].astype(np.int32); bc=fc[kept]
        path=np.empty(ns,np.int32); cb=int(np.argmin(bc))
        for s in range(ns-1,-1,-1): path[s]=bpb[s,cb]; cb=bps[s,cb]
        return tw_tvt[path]


    @njit(cache=False)
    def _dtw_sakoe_chiba(query, ref, radius):
        N = len(query); M = len(ref)
        INF = 1e18
        D = np.full((N, M), INF)
        slope = (M - 1.0) / max(N - 1.0, 1.0)
        for i in range(N):
            j_center = int(round(i * slope))
            j_lo = max(0, j_center - radius)
            j_hi = min(M - 1, j_center + radius)
            for j in range(j_lo, j_hi + 1):
                cost = (query[i] - ref[j]) ** 2
                if i == 0 and j == 0:
                    D[i, j] = cost
                elif i == 0:
                    prev = D[i, j - 1]
                    D[i, j] = cost + (prev if prev < INF else INF)
                elif j == 0:
                    prev = D[i - 1, j]
                    D[i, j] = cost + (prev if prev < INF else INF)
                else:
                    a = D[i - 1, j - 1]
                    b = D[i - 1, j]
                    c = D[i, j - 1]
                    mn = a if a < b else b
                    mn = mn if mn < c else c
                    D[i, j] = cost + (mn if mn < INF else INF)
        i = N - 1; j = M - 1
        pi = np.zeros(N + M, np.int64)
        pj = np.zeros(N + M, np.int64)
        k = 0
        while i > 0 or j > 0:
            pi[k] = i; pj[k] = j; k += 1
            if i == 0:
                j -= 1
            elif j == 0:
                i -= 1
            else:
                a = D[i - 1, j - 1]
                b = D[i - 1, j]
                c = D[i, j - 1]
                if a <= b and a <= c:
                    i -= 1; j -= 1
                elif b <= c:
                    i -= 1
                else:
                    j -= 1
        pi[k] = 0; pj[k] = 0; k += 1
        return D, pi[:k], pj[:k]

    @njit(cache=False)
    def _dtw_path_to_tvt(pi, pj, tw_tvt, N):
        j_for_i = np.zeros(N, np.int64)
        for k in range(len(pi)):
            j_for_i[pi[k]] = pj[k]
        result = np.empty(N, np.float32)
        for i in range(N):
            result[i] = tw_tvt[j_for_i[i]]
        return result

    @njit(cache=False)
    def _dtw_path_slope(pi, pj, N, smooth_win=5):
        j_for_i = np.zeros(N, np.float64)
        for k in range(len(pi)):
            j_for_i[pi[k]] = float(pj[k])
        slope = np.zeros(N, np.float32)
        hw = smooth_win // 2
        for i in range(N):
            i0 = max(0, i - hw); i1 = min(N - 1, i + hw)
            if i1 > i0:
                slope[i] = float((j_for_i[i1] - j_for_i[i0]) / (i1 - i0))
            else:
                slope[i] = 1.0
        return slope

    @njit(cache=False)
    def _dtw_stochastic_realizations(query, ref, radius, K, temperature, seed):
        N = len(query); M = len(ref)
        INF = 1e18
        slope = (M - 1.0) / max(N - 1.0, 1.0)
        D_base = np.full((N, M), INF)
        for i in range(N):
            j_center = int(round(i * slope))
            j_lo = max(0, j_center - radius)
            j_hi = min(M - 1, j_center + radius)
            for j in range(j_lo, j_hi + 1):
                D_base[i, j] = (query[i] - ref[j]) ** 2
        np.random.seed(seed)
        paths = np.zeros((K, N), np.int64)
        for k in range(K):
            D = np.full((N, M), INF)
            for i in range(N):
                j_center = int(round(i * slope))
                j_lo = max(0, j_center - radius)
                j_hi = min(M - 1, j_center + radius)
                for j in range(j_lo, j_hi + 1):
                    u = np.random.random()
                    if u < 1e-12:
                        u = 1e-12
                    if u > 1.0 - 1e-12:
                        u = 1.0 - 1e-12
                    gumbel = -np.log(-np.log(u)) * temperature
                    cost = D_base[i, j] + gumbel
                    if i == 0 and j == 0:
                        D[i, j] = cost
                    elif i == 0:
                        prev = D[i, j - 1]
                        D[i, j] = cost + (prev if prev < INF else INF)
                    elif j == 0:
                        prev = D[i - 1, j]
                        D[i, j] = cost + (prev if prev < INF else INF)
                    else:
                        a = D[i - 1, j - 1]
                        b = D[i - 1, j]
                        c = D[i, j - 1]
                        mn = a if a < b else b
                        mn = mn if mn < c else c
                        D[i, j] = cost + (mn if mn < INF else INF)
            i = N - 1; j = M - 1
            j_for_i = np.zeros(N, np.int64)
            while i > 0 or j > 0:
                j_for_i[i] = j
                if i == 0:
                    j -= 1
                elif j == 0:
                    i -= 1
                else:
                    a = D[i - 1, j - 1]
                    b = D[i - 1, j]
                    c = D[i, j - 1]
                    if a <= b and a <= c:
                        i -= 1; j -= 1
                    elif b <= c:
                        i -= 1
                    else:
                        j -= 1
            j_for_i[0] = 0
            paths[k, :] = j_for_i
        return paths

    def _downsample_for_dtw(values, stride=DTW_STRIDE):
        n = len(values)
        if n == 0:
            return np.array([], dtype=np.int64), np.array([], dtype=np.float32)
        step = max(1, int(stride))
        idx = np.arange(0, n, step, dtype=np.int64)
        if idx[-1] != n - 1:
            idx = np.r_[idx, n - 1].astype(np.int64)
        return idx, np.asarray(values, dtype=np.float32)[idx]

    def _upsample_from_dtw(idx, values, n):
        if n == 0:
            return np.array([], dtype=np.float32)
        if len(idx) == 0 or len(values) == 0:
            return np.full(n, np.nan, dtype=np.float32)
        return np.interp(np.arange(n, dtype=np.float32), idx.astype(np.float32), np.asarray(values, dtype=np.float32)).astype(np.float32)

    def run_dtw_multiscale(query_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII):
        full_n = len(query_gr)
        idx, q = _downsample_for_dtw(query_gr, DTW_STRIDE)
        tw_idx, tw_gr_ds = _downsample_for_dtw(tw_gr, DTW_STRIDE)
        tw_tvt_ds = np.asarray(tw_tvt, dtype=np.float32)[tw_idx] if len(tw_idx) else np.array([], dtype=np.float32)
        N = len(q)
        if full_n == 0 or N == 0 or len(tw_gr_ds) == 0:
            empty = np.array([], dtype=np.float32)
            return {r: empty for r in radii}, {r: empty for r in radii}, {r: np.inf for r in radii}, empty
        qn = (q - np.nanmean(q)) / (np.nanstd(q) + 1e-6)
        rn = (tw_gr_ds - np.nanmean(tw_gr_ds)) / (np.nanstd(tw_gr_ds) + 1e-6)
        qn_f = np.nan_to_num(qn, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
        rn_f = np.nan_to_num(rn, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
        dtw_tvts = {}; dtw_slopes = {}; dtw_costs = {}
        inv_cost_sum = 0.0; tvt_stack = []
        length_gap = abs(len(qn_f) - len(rn_f))
        for r in radii:
            band = int(length_gap + int(r))
            D, pi, pj = _dtw_sakoe_chiba(qn_f, rn_f, band)
            cost = float(D[len(qn_f) - 1, len(rn_f) - 1]) / max(len(qn_f) + len(rn_f), 1)
            tvt_pred_ds = _dtw_path_to_tvt(pi[::-1], pj[::-1], tw_tvt_ds, N)
            slope_ds = _dtw_path_slope(pi[::-1], pj[::-1], N)
            tvt_pred = _upsample_from_dtw(idx, tvt_pred_ds, full_n)
            slope = _upsample_from_dtw(idx, slope_ds, full_n)
            if not np.isfinite(cost):
                cost = 1e9
            dtw_tvts[r] = tvt_pred
            dtw_slopes[r] = slope
            dtw_costs[r] = cost
            ic = 1.0 / (cost + 1e-6)
            inv_cost_sum += ic
            tvt_stack.append((tvt_pred, ic))
        weights = np.array([ic / max(inv_cost_sum, 1e-9) for _, ic in tvt_stack], dtype=np.float32)
        tvts_mat = np.stack([t for t, _ in tvt_stack], axis=1)
        dtw_ens = (tvts_mat * weights[None, :]).sum(axis=1).astype(np.float32)
        return dtw_tvts, dtw_slopes, dtw_costs, dtw_ens

    def run_dtw_stochastic(query_gr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP, seed=SEED):
        full_n = len(query_gr)
        idx, q = _downsample_for_dtw(query_gr, DTW_STRIDE)
        tw_idx, tw_gr_ds = _downsample_for_dtw(tw_gr, DTW_STRIDE)
        tw_tvt_ds = np.asarray(tw_tvt, dtype=np.float32)[tw_idx] if len(tw_idx) else np.array([], dtype=np.float32)
        N = len(q)
        if full_n == 0 or N == 0 or len(tw_gr_ds) == 0:
            empty = np.array([], dtype=np.float32)
            return empty, empty, empty
        qn = ((q - np.nanmean(q)) / (np.nanstd(q) + 1e-6))
        rn = ((tw_gr_ds - np.nanmean(tw_gr_ds)) / (np.nanstd(tw_gr_ds) + 1e-6))
        qn = np.nan_to_num(qn, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
        rn = np.nan_to_num(rn, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float64)
        band = int(abs(len(qn) - len(rn)) + int(radius))
        paths = _dtw_stochastic_realizations(qn, rn, band, int(K), float(temperature), int(seed))
        tvt_realiz = np.empty((K, N), dtype=np.float32)
        for k in range(K):
            tvt_realiz[k, :] = tw_tvt_ds[paths[k, :]].astype(np.float32)
        mean_tvt = _upsample_from_dtw(idx, tvt_realiz.mean(axis=0).astype(np.float32), full_n)
        std_tvt = _upsample_from_dtw(idx, tvt_realiz.std(axis=0).astype(np.float32), full_n)
        cv_tvt = (std_tvt / (np.abs(mean_tvt) + 1e-6)).astype(np.float32)
        return mean_tvt, std_tvt, cv_tvt

    print("Helpers OK ✓")


    # ─ Particle Filters (TVT Z-velocity + ANCC) ──────────────────────

    def _cal_gr_sigma(hw, tw_tvt, tw_gr):
        kn=hw[hw['TVT_input'].notna() & hw['GR'].notna()]
        if len(kn)<20: return PF_GR_SIG_DEF
        ex=np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)
        return np.clip(np.std(kn['GR'].values-ex),PF_GR_SIG_MIN,PF_GR_SIG_MAX)

    def _z_beta(hw):
        kn=hw[hw['TVT_input'].notna()]
        if len(kn)<30: return -1.,0.,0.1
        dz=np.diff(kn['Z'].values); dtvt=np.diff(kn['TVT_input'].values)
        dmd=np.diff(kn['MD'].values); m=dmd>0
        if m.sum()<10: return -1.,0.,0.1
        vz=dz[m]/dmd[m]; vt=dtvt[m]/dmd[m]
        A=np.column_stack([vz,np.ones_like(vz)])
        c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
        return c[0],c[1],max(np.std(vt-(c[0]*vz+c[1])),0.001)

    def _init_v(hw):
        kn=hw[hw['TVT_input'].notna()]
        if len(kn)<10: return 0.
        tail=kn.tail(20); dtvt=np.diff(tail['TVT_input'].values)
        dmd=np.diff(tail['MD'].values); m=dmd>0
        return 0. if m.sum()<3 else float(np.median(dtvt[m]/dmd[m]))

    def run_pf_z(hw, tw_tvt, tw_gr, N=PF_N):
        tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values
        tf_p=interp1d(tw_tvt,tw_gr,bounds_error=False,fill_value=(tw_gr[0],tw_gr[-1]))
        tf_s=interp1d(tw_tvt,tw_s, bounds_error=False,fill_value=(tw_s[0], tw_s[-1]))
        tmin,tmax=tw_tvt.min(),tw_tvt.max()
        gs=_cal_gr_sigma(hw,tw_tvt,tw_gr); beta,icpt,zsig=_z_beta(hw)
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
        pos=float(kn['TVT_input'].iloc[-1])+np.random.normal(0,PF_INIT_SPR,N)
        vel=_init_v(hw)+np.random.normal(0,PF_INIT_V_STD,N)
        w=np.ones(N)/N
        md_v=ev['MD'].values; gr_v=ev['GR'].values; z_v=ev['Z'].values
        pm=float(kn['MD'].iloc[-1]); pz=float(kn['Z'].iloc[-1])
        pts=np.empty(len(ev)); std=np.empty(len(ev))
        for i,idx in enumerate(ev.index):
            dm=max(md_v[i]-pm,1.); dzd=(z_v[i]-pz)/dm
            ve=beta*dzd+icpt
            vel=PF_MOM*vel+np.random.normal(0,PF_VN,N)
            pos=pos+vel*dm+np.random.normal(0,PF_PN,N)
            pos=np.clip(pos,tmin-50,tmax+50)
            if not np.isnan(gr_v[i]):
                ep=tf_p(pos); lp=np.exp(-0.5*((gr_v[i]-ep)/gs)**2)
                gs_sm=gr_sm.iloc[hw.index.get_loc(idx)]
                if not np.isnan(gs_sm):
                    es=tf_s(pos); ls=np.exp(-0.5*((gs_sm-es)/(gs*1.5))**2)
                    lk=(1-PF_GR_WT)*lp+PF_GR_WT*ls
                else: lk=lp
                lk=np.maximum(lk,1e-300); w*=lk; ws=w.sum()
                w=(w/ws) if ws>0 else np.full(N,1./N)
            zs=max(zsig*2.,0.005); lz=np.exp(-0.5*((vel-ve)/zs)**2)
            lz=np.maximum(lz,1e-300); w*=lz; ws=w.sum()
            w=(w/ws) if ws>0 else np.full(N,1./N)
            ne=1./np.sum(w**2)
            if ne<PF_RESAMP*N:
                cum=np.cumsum(w); u=(np.arange(N)+np.random.uniform())/N
                ix=np.searchsorted(cum,u); pos=pos[ix]; vel=vel[ix]; w[:]=1./N
                pos+=np.random.normal(0,PF_ROUGH_P,N); vel+=np.random.normal(0,PF_ROUGH_V,N)
            pts[i]=np.average(pos,weights=w)
            std[i]=np.sqrt(np.average((pos-pts[i])**2,weights=w))
            pm=md_v[i]; pz=z_v[i]
        return pts,std

    def run_pf_ancc(hw, tw_tvt, tw_gr, N=ANCC_N):
        tmin,tmax=tw_tvt.min(),tw_tvt.max()
        gs=_cal_gr_sigma(hw,tw_tvt,tw_gr)
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
        tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values); dz=np.diff(tail['Z'].values)
        dm=np.diff(tail['MD'].values); m=dm>0
        ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
        pos=ls+np.random.normal(0,ANCC_IS,N); rate=ir+np.random.normal(0,ANCC_IR,N)
        w=np.ones(N)/N
        md_v=ev['MD'].values; z_v=ev['Z'].values; gr_v=ev['GR'].values; pm=float(kn['MD'].iloc[-1])
        pts=np.empty(len(ev)); std=np.empty(len(ev))
        for i in range(len(ev)):
            dm=max(md_v[i]-pm,1.)
            rate=ANCC_ALPHA*rate+np.random.normal(0,ANCC_RN,N)
            pos=pos+rate*dm+np.random.normal(0,ANCC_PN,N)
            tvt_e=np.clip(pos-z_v[i],tmin-50,tmax+50); pos=tvt_e+z_v[i]
            if not np.isnan(gr_v[i]):
                eg=np.interp(tvt_e,tw_tvt,tw_gr); lk=np.exp(-0.5*((gr_v[i]-eg)/gs)**2)
                lk=np.maximum(lk,1e-300); w*=lk; ws=w.sum()
                w=(w/ws) if ws>0 else np.full(N,1./N)
            ne=1./np.sum(w**2)
            if ne<PF_RESAMP*N:
                cum=np.cumsum(w); u=(np.arange(N)+np.random.uniform())/N
                ix=np.searchsorted(cum,u); pos=pos[ix]; rate=rate[ix]; w[:]=1./N
                pos+=np.random.normal(0,ANCC_RP,N); rate+=np.random.normal(0,ANCC_RR,N)
            tv=float(np.average(pos-z_v[i],weights=w)); pts[i]=tv
            std[i]=np.sqrt(np.average((pos-z_v[i]-tv)**2,weights=w))
            pm=md_v[i]
        return pts,std

    print("Particle Filters OK ✓")


    # ─ Spatial Imputers ──────────────────────────────────────────────
    # FormationPlaneKNN: full 6-formation plane-fit (ANCC + 5 others)
    # DenseANCCImputer: 60 pts/well IDW for fine spatial resolution

    class FormationPlaneKNN:
        def __init__(self, well_ids, data_dir):
            rows=[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
                except: continue
                if len(df)==0: continue
                row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
                for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
                rows.append(row)
            self.df=pd.DataFrame(rows)
            self.wmap={w:i for i,w in enumerate(self.df['wid'])}
            xy=self.df[['x','y']].to_numpy()
            self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
            self.tree=cKDTree(xy/self.scale)
            self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
            self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

        def impute(self, xy_q, self_wid=None, k=PLANE_K):
            q=xy_q/self.scale; nf=min(k+5,len(self.df))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
            xn=self.xa[ik]; yn=self.ya[ik]
            wx=w*xn; wy=w*yn
            A=np.zeros((len(q),3,3))
            A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
            A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
            A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
            A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
            fn=self.fa[ik]   # (N,K,6)
            rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
            try: coef=np.linalg.solve(A,rhs)
            except:
                coef=np.zeros((len(q),3,6))
                for r in range(len(q)):
                    try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                    except: pass
            Xq=xy_q[:,0]; Yq=xy_q[:,1]
            pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
            pred[~vk.any(1)]=self.fa.mean(0)
            return pred, np.where(vk,dk,np.inf).min(1).astype(np.float32)

    class DenseANCCImputer:
        def __init__(self, well_ids, data_dir, spw=DENSE_SPW):
            xs,ys,anccs,wids=[],[],[],[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
                except: continue
                if len(df)==0: continue
                ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int)
                s=df.iloc[ix]
                xs.append(s['X'].values); ys.append(s['Y'].values)
                anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
            self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
            self.ancc=np.concatenate(anccs).astype(np.float32)
            self.wids=np.array(wids)
            self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
            self.tree=cKDTree(self.xy/self.scale)

        def impute(self, xy_q, self_wid=None, k=DENSE_K, nfetch=3000):
            xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale
            nf=min(nfetch,len(self.ancc))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
            sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw)
            an=self.ancc[ik]
            ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
            var=((an-ap[:,None])**2*w).sum(1)/safe
            return (ap.astype(np.float32),
                    np.sqrt(np.maximum(var,0.)).astype(np.float32),
                    np.where(vk,dk,np.inf).min(1).astype(np.float32))

    if not bool(globals().get('RUN_FAST_PF_SELECTOR_ONLY', False)):
    # Build imputers
        hw_paths=sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
        train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
        print(f"Building imputers from {len(train_wids)} wells...")
        t0=time.time()
        FI=FormationPlaneKNN(train_wids,TRAIN_DIR)
        DI=DenseANCCImputer(train_wids,TRAIN_DIR)
        print(f"  FormationPF: {len(FI.df)} centroids | DenseANCC: {len(DI.ancc):,} pts  ({time.time()-t0:.0f}s)")


        # ─ Feature Builder (per well, global FI/DI, thread-safe) ──────────
        _FI=FI; _DI=DI

        ANCH_OFFS = np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],dtype=np.float32)
        BEAM_OFFS = np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],  dtype=np.float32)
        SC_OFFS   = np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],    dtype=np.float32)
        DTW_OFFS  = np.array([-20,-10,-5,-2,0,2,5,10,20],          dtype=np.float32)

        def build_well(hw_path, tw_path, is_train):
            global _FI,_DI
            wid=Path(hw_path).stem.replace('__horizontal_well','')
            well_seed = stable_seed(wid, SEED)
            np.random.seed(well_seed)
            try:
                hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
            except: return None
            if is_train and 'TVT' not in hw.columns: return None
            kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
            if len(ev)==0 or len(kn)<10: return None
            if is_train and hw['TVT'].isna().all(): return None

            tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
            if len(tw_tvt)<3: return None

            # PF signals (use ANCC PF as primary)
            pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
            if len(pf_a)==0: return None
            pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
            pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
            has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))

            # Beam search (5 configs)
            lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
            gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
            hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
            kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)
            bpaths={}
            for (bs,mc,es,r,tag) in BEAMS:
                bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
            beam_ref=(bpaths['cons']+bpaths['sm5'])/2.

            # Self-correlation
            ktvt=kn['TVT_input'].to_numpy(np.float32)
            sc_raw,sc_sc=self_corr_tvt(kgr,ktvt,hgr,hw=15,stride=3)
            sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
            hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_raw

            # Constrained / stochastic DTW over the full horizontal GR sequence.
            full_gr = gr_full.values.astype(np.float32)
            dtw_tvts_ms, dtw_slopes_ms, dtw_costs_ms, dtw_ens_ms = run_dtw_multiscale(
                full_gr, tw_tvt, tw_gr, last_tvt, radii=DTW_RADII
            )
            stoch_seed = stable_seed(wid, SEED + 2607)
            dtw_mean_stoch, dtw_std_stoch, dtw_cv_stoch = run_dtw_stochastic(
                full_gr, tw_tvt, tw_gr, last_tvt, radius=50, K=DTW_STOCH_K, temperature=DTW_STOCH_TEMP, seed=stoch_seed
            )
            nh=len(ev); ev_start=int(ev.index[0])
            dtw_ens_raw_ms = dtw_ens_ms.copy()
            dtw_anchor_error = np.float32(dtw_ens_raw_ms[ev_start] - np.float32(last_tvt)) if len(dtw_ens_raw_ms) > ev_start else np.float32(0.0)
            dtw_ens_ms = (dtw_ens_raw_ms - dtw_anchor_error).astype(np.float32)
            for r in DTW_RADII:
                if len(dtw_tvts_ms[r]) > ev_start:
                    shift_r = np.float32(dtw_tvts_ms[r][ev_start] - np.float32(last_tvt))
                    dtw_tvts_ms[r] = (dtw_tvts_ms[r] - shift_r).astype(np.float32)
            dtw_stoch_anchor_error = np.float32(dtw_mean_stoch[ev_start] - np.float32(last_tvt)) if len(dtw_mean_stoch) > ev_start else np.float32(0.0)
            dtw_mean_stoch = (dtw_mean_stoch - dtw_stoch_anchor_error).astype(np.float32)
            def _ev_slice(arr):
                return np.asarray(arr[ev_start:ev_start+nh], dtype=np.float32)
            dtw_ens_raw_ev = _ev_slice(dtw_ens_raw_ms)
            dtw_ens_ev = _ev_slice(dtw_ens_ms)
            dtw_mean_ev = _ev_slice(dtw_mean_stoch)
            dtw_std_ev = _ev_slice(dtw_std_stoch)
            dtw_cv_ev = _ev_slice(dtw_cv_stoch)
            dtw_per_radius_ev = {r: _ev_slice(dtw_tvts_ms[r]) for r in DTW_RADII}
            dtw_slope_ev = {r: _ev_slice(dtw_slopes_ms[r]) for r in DTW_RADII}
            dtw_slope_mean_ev = np.stack([dtw_slope_ev[r] for r in DTW_RADII], 1).mean(1).astype(np.float32)
            dtw_cost_arr = np.array([dtw_costs_ms[r] for r in DTW_RADII], dtype=np.float32)
            dtw_cost_min = float(np.nanmin(dtw_cost_arr))
            dtw_cost_range = float(np.nanmax(dtw_cost_arr) - np.nanmin(dtw_cost_arr))

            # Affine calibration
            tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
            a_cal,b_cal=affine_cal(kgr,tw_at_k)

            # Prefix stats
            kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
            pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
            slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
            slp_z=robust_slope(kz,ktvt)

            # Spatial ANCC (centroid plane-fit)
            swid=wid if is_train else None
            xy_ev=ev[['X','Y']].to_numpy(np.float64)
            xy_kn=kn[['X','Y']].to_numpy(np.float64)
            form_ev, knn_d=_FI.impute(xy_ev,self_wid=swid)   # (nh,6)
            form_kn,_     =_FI.impute(xy_kn,self_wid=swid)

            # b_well per formation + TVT formula
            z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)
            tvt_formulas={}
            for fi2,fn in enumerate(FORMATIONS):
                b_v=ktvt+z_kn-form_kn[:,fi2]
                b_all=float(np.median(b_v)); b_50=float(np.median(b_v[-50:])) if len(b_v)>=5 else b_all
                tvt_formulas[f'tvtF_{fn}']=(-z_ev+form_ev[:,fi2]+b_all).astype(np.float32)
                tvt_formulas[f'tvtF50_{fn}']=(-z_ev+form_ev[:,fi2]+b_50).astype(np.float32)
                tvt_formulas[f'bw_{fn}']=np.float32(b_all)
                tvt_formulas[f'bw50_{fn}']=np.float32(b_50)

            # Dense ANCC
            d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
            d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
            b_vd=ktvt+z_kn-d_kn
            b_d=float(np.median(b_vd)); b_d50=float(np.median(b_vd[-50:])) if len(b_vd)>=5 else b_d
            tvt_dense=(-z_ev+d_ancc+b_d).astype(np.float32)
            tvt_dense50=(-z_ev+d_ancc+b_d50).astype(np.float32)
            # Dense reliability in the known prefix should measure residual spread
            # around the fitted well offset, not the absolute offset magnitude.
            dense_offset_resid=(b_vd-b_d).astype(np.float32)
            d_rmse=float(np.sqrt(np.mean(dense_offset_resid**2)))
            d_bias=float(np.mean(dense_offset_resid)); d_nb_std=float(np.mean(d_std_kn))
            last_form_ancc=float(form_kn[-1,0]) if len(form_kn) else float(np.nanmean(form_ev[:,0]))

            # GR rolling features (multiple scales)
            gr_s=pd.Series(gr_full.values)
            rolls={}
            for w in [5,21,51,101]:
                r=gr_s.rolling(w,center=True,min_periods=1)
                rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
                rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
            for lag in [1,5,15,30]:
                rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
                rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
            gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
            gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)

            # Slope baselines
            hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
            slp_base_all=(last_tvt+slp_all*md_since).astype(np.float32)
            slp_base_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

            # Trajectory
            mdd=hw['MD'].diff().replace(0,np.nan)
            dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
            dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
            dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

            frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
            def sc(v): return np.full(nh,np.float32(v),np.float32)

            feats={
                'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
                'last_known_tvt':sc(last_tvt),
                # PF signals
                'pf_ancc':pf_use,'pf_ancc_std':std_use,
                'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
                'pf_z':pf_z.astype(np.float32) if has_z else sc(last_tvt),
                'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
                'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
                # Beam paths (5)
                **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
                'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
                'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
                'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
                # Self-corr
                'sc_d':(sc_raw-np.float32(last_tvt)).astype(np.float32),'sc_score':sc_sc,'sc_trust':sc(sc_trust),
                'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
                # DTW sequence alignment
                'dtw_ens_d_raw':(dtw_ens_raw_ev-np.float32(last_tvt)).astype(np.float32),
                'dtw_ens_d':(dtw_ens_ev-np.float32(last_tvt)).astype(np.float32),
                'dtw_anchor_error':sc(dtw_anchor_error),
                'dtw_anchor_abs_error':sc(abs(float(dtw_anchor_error))),
                'dtw_stoch_anchor_error':sc(dtw_stoch_anchor_error),
                'dtw_stoch_mean_d':(dtw_mean_ev-np.float32(last_tvt)).astype(np.float32),
                'dtw_stoch_std':dtw_std_ev,
                'dtw_stoch_cv':dtw_cv_ev,
                'dtw_slope_mean':dtw_slope_mean_ev,
                **{f'dtw_r{int(r)}_d':(dtw_per_radius_ev[r]-np.float32(last_tvt)).astype(np.float32) for r in DTW_RADII},
                **{f'dtw_slope_r{int(r)}':dtw_slope_ev[r] for r in DTW_RADII},
                'dtw_cost_min':sc(dtw_cost_min),
                'dtw_cost_range':sc(dtw_cost_range),
                'dtw_vs_beam':(dtw_ens_ev-beam_ref).astype(np.float32),
                'dtw_vs_pf':(dtw_ens_ev-pf_use).astype(np.float32),
                'dtw_vs_sc':(dtw_ens_ev-sc_raw).astype(np.float32),
                # Spatial / formula
                **tvt_formulas,
                'spatial_ancc_d':(form_ev[:,0]-np.float32(last_form_ancc)).astype(np.float32),
                'spatial_knn_dist':knn_d,
                # Dense ANCC
                'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
                'tvt_dense_d':(tvt_dense-last_tvt).astype(np.float32),
                'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
                'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
                # PF vs spatial/dense
                'pf_vs_spatial':(pf_use-tvt_formulas['tvtF_ANCC']).astype(np.float32),
                'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
                'spatial_vs_dense':(tvt_formulas['tvtF_ANCC']-tvt_dense).astype(np.float32),
                'beam_vs_spatial':(bpaths['cons']-tvt_formulas['tvtF_ANCC']).astype(np.float32),
                'dtw_vs_dense':(dtw_ens_ev-tvt_dense).astype(np.float32),
                'dtw_vs_form':(dtw_ens_ev-tvt_formulas['tvtF_ANCC']).astype(np.float32),
                # Affine cal
                'cal_a':sc(a_cal),'cal_b':sc(b_cal),
                # Prefix stats
                'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
                'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
                'slp_base_d_all':(slp_base_all-last_tvt).astype(np.float32),
                'slp_base_d_50': (slp_base_50 -last_tvt).astype(np.float32),
                'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
                # Position
                'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
                'z':z_ev,
                'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
                'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
                'dz':(z_ev-float(lk['Z'])).astype(np.float32),
                'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
                'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
                # GR row
                'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,
                'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
                'gr_vs_slp_all':hgr-np.interp(slp_base_all,tw_tvt,tw_gr).astype(np.float32),
                # tw_diff 3 families
                **{f'tda{int(o)}':hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
                **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
                **{f'tdsc{int(o)}':hgr-np.interp(sc_raw+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
                **{f'tddtw{int(o)}':hgr-np.interp(dtw_ens_ev+o,tw_tvt,tw_gr).astype(np.float32) for o in DTW_OFFS},
                # Typewell stats
                'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
            }
            for k,v in rolls.items(): feats[k]=v

            result=pd.DataFrame(feats)
            if is_train:
                if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
                result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
            return result


        def build_dataset(paths, is_train, label):
            args=[(str(p), str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'), is_train)
                  for p in paths
                  if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
            print(f"  {label}: {len(args)} wells | {NCPU} threads")
            res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
                delayed(build_well)(hp,tp,it) for hp,tp,it in args)
            parts=[r for r in res if r is not None]
            print(f"  {label}: OK={len(parts)} skipped={len(args)-len(parts)}")
            return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

        print("Feature builder OK ✓")


    

# Target-free PF/beam selector candidate. This block mirrors the physical-model PF selector reference.
    PF_SELECTOR_BIN_VARIANTS = {
        0: "pf_scale_5_hold_0.2",
        1: "pf_scale_3_hold_0.15",
        2: "pf_scale_12_beam_0.2_hold_0.15",
        3: "pf_scale_5_hold_0.15",
        4: "pf_scale_5_beam_0.05_hold_0.05",
        5: "pf_scale_12_beam_0.2_hold_0.05",
    }
    PF_SELECTOR_GLOBAL_VARIANT = "pf_scale_8_hold_0.2"
    PF_SELECTOR_N_EVAL_THRESHOLD = 4840.0
    PF_SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)
    PF_SELECTOR_BEAM_CONFIGS = [
        (10, 20.0, 144.0, 2),
        (10,  8.0,  64.0, 2),
        ( 8, 35.0, 220.0, 1),
        (10, 14.0,  90.0, 5),
        (20,  4.0,  36.0, 3),
        (12, 12.0, 100.0, 3),
        (15, 25.0, 180.0, 2),
        (20, 30.0, 200.0, 2),
        (15, 10.0,  80.0, 4),
        (25,  6.0,  50.0, 3),
        (10, 40.0, 300.0, 1),
        (12, 18.0, 120.0, 5),
        (30,  8.0,  70.0, 2),
        (10, 50.0, 400.0, 0),
    ]

    def _selector_tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
        tw_g = tw_tr.dropna(subset=['Geology'])
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
        if np.isnan(ref_tvt):
            ref_col = tw_g['Geology'].iloc[0]
            ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
        offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
        return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset

    def _selector_particle_filter(hw, tw, n_particles=500, seed=42):
        tw_s = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(ev) == 0:
            return hw['TVT_input'].values.astype(float).copy(), 0.0

        last = kn.iloc[-1]
        last_tvt = float(last['TVT_input'])
        last_Z = float(last['Z'])
        last_MD = float(last['MD'])

        tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
        gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10.0, 60.0))

        tail = kn.tail(30)
        dt = np.diff(tail['TVT_input'].values)
        dz = np.diff(tail['Z'].values)
        dm = np.diff(tail['MD'].values)
        m = dm > 0
        ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

        N = int(n_particles)
        rng = np.random.default_rng(seed)
        ls = last_tvt + last_Z
        pos = ls + 2.0 * rng.standard_normal(N)
        rate = ir + 0.01 * rng.standard_normal(N)
        w = np.ones(N) / N

        MOM = 0.998
        VN = 0.002
        PN = 0.005
        RP = 0.1
        RR = 0.001
        RESAMP = 0.5

        md_v = ev['MD'].values.astype(float)
        z_v = ev['Z'].values.astype(float)
        gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
        gr_v = gr_interp.values.astype(float)[ev.index]

        out_vals = hw['TVT_input'].values.astype(float).copy()
        res = np.empty(len(ev))
        prev_MD = last_MD
        log_lik = 0.0

        for i in range(len(ev)):
            dm_step = max(md_v[i] - prev_MD, 1.0)
            rate = MOM * rate + VN * rng.standard_normal(N)
            pos = pos + rate * dm_step + PN * rng.standard_normal(N)
            tvt_p = pos - z_v[i]
            tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
            pos = tvt_p + z_v[i]

            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr_v[i] - eg) / gs
            lk = np.exp(-0.5 * np.minimum(d**2, 600.0))
            lk = np.maximum(lk, 1e-300)
            avg_lk = float((w * lk).sum())
            log_lik += np.log(max(avg_lk, 1e-300))
            w = w * lk
            ws = w.sum()
            w = w / ws if ws > 0 else np.ones(N) / N

            n_eff = 1.0 / (w**2).sum()
            if n_eff < RESAMP * N:
                cum = np.cumsum(w)
                u0 = rng.uniform(0, 1.0 / N)
                idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
                pos = pos[idx] + RP * rng.standard_normal(N)
                rate = rate[idx] + RR * rng.standard_normal(N)
                w = np.ones(N) / N

            res[i] = float(np.dot(w, pos - z_v[i]))
            prev_MD = md_v[i]

        out_vals[list(ev.index)] = res
        return out_vals, log_lik

    def _selector_pf_scales(hw, tw, scales, n_particles=500, n_seeds=64):
        preds = []
        liks = []
        for seed in range(int(n_seeds)):
            pred, ll = _selector_particle_filter(hw, tw, n_particles=n_particles, seed=seed)
            preds.append(pred)
            liks.append(ll)
        pred_arr = np.stack(preds, 0)
        liks = np.array(liks)
        liks_n = liks - liks.max()
        out = {}
        for scale in scales:
            weights = np.exp(liks_n / float(scale))
            weights /= weights.sum()
            out[f"pf_scale_{scale:g}"] = (weights[:, None] * pred_arr).sum(0)
        out["pf_mean"] = pred_arr.mean(0)
        return out

    def _selector_beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
        n = len(hgr)
        nt = len(tw_tvt)
        if n == 0:
            return np.array([last_tvt])

        if r > 0 and n > max(3, 2 * r + 1):
            win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()

        si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
        MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
        MC = mc * np.array([2.0, 1.0, 0.0, 1.0, 2.0])

        bidx = np.full(bs, si, dtype=np.int64)
        bcost = np.full(bs, np.inf)
        bcost[0] = 0.0
        bn = 1
        result = np.zeros(n)

        for step in range(n):
            gv = sgr[step]
            ni = bidx[:bn, None] + MOVES[None, :]
            ci = np.clip(ni, 0, nt - 1)
            valid = (ni >= 0) & (ni < nt)

            gr_e = (gv - tw_gr[ci])**2 / es
            tot = bcost[:bn, None] + gr_e + MC[None, :]
            tot = np.where(valid, tot, np.inf)

            ni_f = ni.flatten()
            tot_f = tot.flatten()
            vf = valid.flatten()
            ni_f = ni_f[vf]
            tot_f = tot_f[vf]

            order = np.argsort(tot_f)
            ni_s = ni_f[order]
            tot_s = tot_f[order]

            _, first = np.unique(ni_s, return_index=True)
            ni_u = ni_s[first]
            tot_u = tot_s[first]

            kept = min(bs, len(ni_u))
            top = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
            top = top[np.argsort(tot_u[top])]

            bidx[:kept] = ni_u[top]
            bcost[:kept] = tot_u[top]
            if kept < bs:
                bidx[kept:] = bidx[kept - 1]
                bcost[kept:] = np.inf
            bn = kept

            result[step] = tw_tvt[bidx[0]]

        return result

    def _selector_beam_ensemble(hw, tw):
        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(ev) == 0:
            return hw['TVT_input'].values.astype(float).copy()

        last_tvt = float(kn.iloc[-1]['TVT_input'])
        tw_s = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

        gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
        hgr = gr_all[ev.index]

        beam_results = [
            _selector_beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
            for (bs, mc, es, r) in PF_SELECTOR_BEAM_CONFIGS
        ]
        beam_mean = np.stack(beam_results, 0).mean(0)

        out = hw['TVT_input'].values.astype(float).copy()
        out[list(ev.index)] = beam_mean
        return out

    def _selector_well_code(hw):
        eval_mask = hw['TVT_input'].isna().to_numpy()
        n_eval = float(eval_mask.sum())
        z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
        z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
        n_bin = int(n_eval > PF_SELECTOR_N_EVAL_THRESHOLD)
        z_bin = int(np.searchsorted(PF_SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
        code = n_bin + 2 * z_bin
        return code, PF_SELECTOR_BIN_VARIANTS.get(code, PF_SELECTOR_GLOBAL_VARIANT), n_eval, z_span

    def _selector_parse_variant(name):
        parts = name.split('_')
        scale = float(parts[2])
        beam_weight = 0.0
        hold_weight = 0.0
        if 'beam' in parts:
            beam_weight = float(parts[parts.index('beam') + 1])
        if 'hold' in parts:
            hold_weight = float(parts[parts.index('hold') + 1])
        return scale, beam_weight, hold_weight

    def _selector_apply_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
        scale, beam_weight, hold_weight = _selector_parse_variant(name)
        base = pf_by_scale.get(f"pf_scale_{scale:g}")
        if base is None:
            base = pf_by_scale[PF_SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
        pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
        pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
        return pred

    def _build_target_free_selector_submission(sample_df):
        scales = tuple(float(s) for s in globals().get('PF_SELECTOR_SCALES', (3.0, 5.0, 8.0, 12.0)))
        sample_work = sample_df[['id']].copy()
        sample_work['_well'] = sample_work['id'].astype(str).str[:8]
        sample_work['_row_idx'] = sample_work['id'].astype(str).str[9:].astype(int)
        train_wids = {path.stem.replace('__horizontal_well', '') for path in TRAIN_DIR.glob('*__horizontal_well.csv')}
        rows, report_rows = [], []
        for wid in sorted(sample_work['_well'].unique()):
            hw_path = TEST_DIR / f'{wid}__horizontal_well.csv'
            tw_path = TEST_DIR / f'{wid}__typewell.csv'
            if not hw_path.exists() or not tw_path.exists():
                raise FileNotFoundError(f'Missing selector input files for well {wid}')
            hw = pd.read_csv(hw_path)
            tw = pd.read_csv(tw_path)
            tvt_phys = None
            tw_ref = tw
            if bool(globals().get('PF_SELECTOR_USE_SAME_WELL_PHYSICAL', False)) and wid in train_wids:
                try:
                    hw_tr = pd.read_csv(TRAIN_DIR / f'{wid}__horizontal_well.csv')
                    tw_tr = pd.read_csv(TRAIN_DIR / f'{wid}__typewell.csv')
                    hw['TVT_input'] = hw_tr['TVT_input'].values
                    tvt_phys = _selector_tvt_from_contacts(hw_tr, tw_tr)
                    tw_ref = tw_tr
                except Exception as exc:
                    report_rows.append({'well_id': wid, 'stage': 'physical_fallback', 'message': str(exc)[:200]})
                    tvt_phys = None
                    tw_ref = tw
            code, variant, n_eval, z_span = _selector_well_code(hw)
            try:
                pf_by_scale = _selector_pf_scales(
                    hw, tw_ref, scales,
                    n_particles=int(globals().get('PF_SELECTOR_N_PARTICLES', 500)),
                    n_seeds=int(globals().get('PF_SELECTOR_N_SEEDS', 64)),
                )
            except Exception as exc:
                last_known = hw['TVT_input'].dropna()
                last_val = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
                tvt_pf = hw['TVT_input'].fillna(last_val).values.astype(float)
                pf_by_scale = {f"pf_scale_{scale:g}": tvt_pf.copy() for scale in scales}
                report_rows.append({'well_id': wid, 'stage': 'pf_fallback', 'message': str(exc)[:200]})
            try:
                tvt_beam = _selector_beam_ensemble(hw, tw_ref)
            except Exception as exc:
                tvt_beam = pf_by_scale.get('pf_scale_8', next(iter(pf_by_scale.values()))).copy()
                report_rows.append({'well_id': wid, 'stage': 'beam_fallback', 'message': str(exc)[:200]})
            last_known = hw['TVT_input'].dropna()
            last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(pf_by_scale.get('pf_scale_8', next(iter(pf_by_scale.values())))))
            tvt_selector = _selector_apply_variant(variant, pf_by_scale, tvt_beam, last_known_tvt)
            ws = sample_work[sample_work['_well'] == wid]
            for _, row in ws.iterrows():
                ridx = int(row['_row_idx'])
                if tvt_phys is not None:
                    tvt_val = float(tvt_phys.iloc[ridx])
                else:
                    tvt_val = float(tvt_selector[ridx])
                rows.append({'id': row['id'], 'tvt': tvt_val})
            report_rows.append({
                'well_id': wid,
                'stage': 'selector',
                'selector_code': int(code),
                'selector_variant': variant,
                'n_eval': float(n_eval),
                'z_span': float(z_span),
                'rows': int(len(ws)),
                'used_same_well_physical': bool(tvt_phys is not None),
            })
        out = sample_df[['id']].merge(pd.DataFrame(rows), on='id', how='left')
        if out['tvt'].isna().any():
            bad = out.loc[out['tvt'].isna(), 'id'].head(10).tolist()
            raise RuntimeError(f'Target-free selector missing ids: {bad}')
        if not np.isfinite(out['tvt'].to_numpy(dtype=float)).all():
            raise RuntimeError('Target-free selector produced non-finite TVT values.')
        pd.DataFrame(report_rows).to_csv(OUTPUT_DIR / 'target_free_selector_report.csv', index=False)
        return out[['id', 'tvt']]

    def _run_fast_target_free_selector_submission():
        sample = pd.read_csv(SAMPLE)
        selector_sub = _build_target_free_selector_submission(sample)
        selector_sub = sample[['id']].merge(selector_sub, on='id', how='left')
        if selector_sub['tvt'].isna().any():
            bad = selector_sub.loc[selector_sub['tvt'].isna(), 'id'].head(10).tolist()
            raise RuntimeError(f'Fast selector missing sample ids: {bad}')
        selector_sub['tvt'] = pd.to_numeric(selector_sub['tvt'], errors='coerce')
        if not np.isfinite(selector_sub['tvt'].to_numpy(dtype=float)).all():
            raise RuntimeError('Fast selector produced non-finite TVT values.')
        pf_selector_output = OUTPUT_DIR / 'submission_pf_selector.csv'
        selector_sub[['id', 'tvt']].to_csv(pf_selector_output, index=False)
        selector_sub[['id', 'tvt']].to_csv(OUT, index=False)
        globals()['FINAL_SELECTED_BASE_SOURCE'] = pf_selector_output
        globals()['FINAL_BASE_SOURCE_LABEL'] = str(globals().get('SUBMISSION_PROFILE', 'fast_pf_selector'))

        candidate_selection_summary = pd.DataFrame([{
            'candidate': 'pf_selector',
            'selected': True,
            'oof_rmse_used_for_selection': np.nan,
            'tvt_mean': float(selector_sub['tvt'].mean()),
            'tvt_std': float(selector_sub['tvt'].std()),
            'tvt_min': float(selector_sub['tvt'].min()),
            'tvt_max': float(selector_sub['tvt'].max()),
        }])
        candidate_selection_summary.to_csv(OUTPUT_DIR / 'v7_candidate_selection_summary.csv', index=False)
        pd.DataFrame([{
            'final_source': str(globals().get('SUBMISSION_PROFILE', 'fast_pf_selector')),
            'final_output': str(OUT),
            'selector_output': str(pf_selector_output),
            'final_candidate_requested': str(globals().get('FINAL_V7_CANDIDATE', 'pf_selector')),
            'final_candidate_selected': 'pf_selector',
            'submission_rows': int(len(selector_sub)),
            'submission_tvt_mean': float(selector_sub['tvt'].mean()),
            'submission_tvt_std': float(selector_sub['tvt'].std()),
            'submission_tvt_min': float(selector_sub['tvt'].min()),
            'submission_tvt_max': float(selector_sub['tvt'].max()),
            'pf_selector_n_particles': int(globals().get('PF_SELECTOR_N_PARTICLES', 500)),
            'pf_selector_n_seeds': int(globals().get('PF_SELECTOR_N_SEEDS', 64)),
            'pf_selector_scales_json': json.dumps([float(s) for s in globals().get('PF_SELECTOR_SCALES', (3.0, 5.0, 8.0, 12.0))]),
            'pf_selector_use_same_well_physical': bool(globals().get('PF_SELECTOR_USE_SAME_WELL_PHYSICAL', False)),
        }]).to_csv(OUTPUT_DIR / 'submission_contract_guard_summary_v7.csv', index=False)
        print(f"\n✅  {OUT}  {len(selector_sub)} rows")
        print(f"Final candidate: pf_selector ({globals().get('SUBMISSION_PROFILE', 'fast_pf_selector')})")
        display(candidate_selection_summary)
        display(selector_sub.head(8))
        return selector_sub[['id', 'tvt']]

    if bool(globals().get('RUN_FAST_PF_SELECTOR_ONLY', False)):
        sub = _run_fast_target_free_selector_submission()
    else:
        # ─ Load Data ──────────────────────────────────────────────────────
        print("Building train..."); t0=time.time()
        train_df=build_dataset(hw_paths,is_train=True,label="train")
        print(f"  train: {train_df.shape}  ({time.time()-t0:.0f}s)")

        test_paths=sorted(TEST_DIR.glob('*__horizontal_well.csv'))
        print("Building test..."); t0=time.time()
        test_df=build_dataset(test_paths,is_train=False,label="test")
        print(f"  test: {test_df.shape}  ({time.time()-t0:.0f}s)")

        SKIP={'well','id','target'}
        feature_cols=[c for c in train_df.columns if c not in SKIP]
        print(f"#features: {len(feature_cols)}")

        X=train_df[feature_cols].replace([np.inf, -np.inf], np.nan).astype(np.float32)
        y=train_df['target'].astype(np.float32)
        g=train_df['well']
        Xt=test_df[feature_cols].replace([np.inf, -np.inf], np.nan).astype(np.float32)
        train_matrix_mb = X.memory_usage(deep=True).sum() / 1e6
        test_matrix_mb = Xt.memory_usage(deep=True).sum() / 1e6
        print(f"Train matrix memory MB: {train_matrix_mb:.1f}")
        print(f"Test matrix memory MB: {test_matrix_mb:.1f}")
        gc.collect()


        # ─ Training: LGB×3 seeds + CatBoost, GroupKFold(5), Ridge + hill stacks ──
        cv=GroupKFold(n_splits=N_SPLITS)
        splits=list(cv.split(X,y,g))

        fold_rows=[]

        def run_lgb(seed):
            p=dict(LGB_P,n_estimators=5000,seed=seed)
            oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
            for fold,(tr,va) in enumerate(splits):
                dtr=lgb.Dataset(X.iloc[tr],label=y.iloc[tr])
                dva=lgb.Dataset(X.iloc[va],label=y.iloc[va],reference=dtr)
                m=lgb.train(p,dtr,valid_sets=[dva],num_boost_round=p['n_estimators'],
                            callbacks=[lgb.early_stopping(150,verbose=False),lgb.log_evaluation(500)])
                oof[va]=m.predict(X.iloc[va],num_iteration=m.best_iteration).astype(np.float32)
                tp+=m.predict(Xt,num_iteration=m.best_iteration).astype(np.float32)/N_SPLITS
                fold_rmse = root_mean_squared_error(y.iloc[va], oof[va])
                fold_rows.append({'model': f'lgb{seed}', 'fold': int(fold + 1), 'rmse': float(fold_rmse), 'best_iteration': int(m.best_iteration)})
                print(f"   LGB{seed} fold{fold}: rmse={fold_rmse:.4f} iter={m.best_iteration}")
            r=root_mean_squared_error(y,oof); print(f"   LGB{seed} OOF={r:.4f}"); return oof,tp,r

        def run_cb():
            p=dict(CB_P)
            oof=np.zeros(len(train_df),np.float32); tp=np.zeros(len(test_df),np.float32)
            for fold,(tr,va) in enumerate(splits):
                m=CatBoostRegressor(**p)
                m.fit(Pool(X.iloc[tr].values,label=y.iloc[tr].values),
                      eval_set=Pool(X.iloc[va].values,label=y.iloc[va].values),use_best_model=True)
                oof[va]=m.predict(X.iloc[va].values).astype(np.float32)
                tp+=m.predict(Xt.values).astype(np.float32)/N_SPLITS
                fold_rmse = root_mean_squared_error(y.iloc[va], oof[va])
                best_iter = getattr(m, 'best_iteration_', None)
                fold_rows.append({'model': 'cb', 'fold': int(fold + 1), 'rmse': float(fold_rmse), 'best_iteration': int(best_iter) if best_iter is not None else np.nan})
                print(f"   CB fold{fold}: rmse={fold_rmse:.4f}")
            r=root_mean_squared_error(y,oof); print(f"   CB OOF={r:.4f}"); return oof,tp,r

        results={}
        for s in LGB_SEEDS:
            oof,tp,r=run_lgb(s); results[f'lgb{s}']={'oof':oof,'test':tp,'rmse':r}
        oof,tp,r=run_cb(); results['cb']={'oof':oof,'test':tp,'rmse':r}

        # Stack candidates: best single, simple average, positive ridge, and sparse hill-climb.
        stack_names=list(results.keys())
        Sx=np.column_stack([results[k]['oof'] for k in stack_names])
        St=np.column_stack([results[k]['test'] for k in stack_names])
        y_arr=y.values.astype(np.float32)

        ridge=Ridge(alpha=1.,fit_intercept=False,positive=True)
        ridge.fit(Sx,y_arr)
        oof_s=ridge.predict(Sx).astype(np.float32); test_s=ridge.predict(St).astype(np.float32)
        r_avg=root_mean_squared_error(y_arr,Sx.mean(1))
        r_stk=root_mean_squared_error(y_arr,oof_s)
        wts=ridge.coef_/max(ridge.coef_.sum(),1e-9)

        def _rmse_np(yv, pv):
            diff=yv.astype(np.float32)-pv.astype(np.float32)
            return float(np.sqrt(np.mean(diff*diff)))

        def hill_climb_stack(result_dict, yv, max_rounds=6):
            names=list(result_dict.keys())
            scores={name:_rmse_np(yv,result_dict[name]['oof']) for name in names}
            best_name=min(scores,key=scores.get)
            cur_oof=result_dict[best_name]['oof'].astype(np.float32).copy()
            cur_test=result_dict[best_name]['test'].astype(np.float32).copy()
            weights={best_name:1.0}
            best_score=scores[best_name]
            grid=np.array([0.01,0.02,0.03,0.05,0.08,0.10,0.15,0.20,0.25,0.30,0.35,0.40],dtype=np.float32)
            trace=[{'round':0,'added_model':best_name,'weight':1.0,'rmse':best_score}]
            for rd in range(1,max_rounds+1):
                step=None
                for name in names:
                    cand_oof=result_dict[name]['oof'].astype(np.float32)
                    for w in grid:
                        trial=(1.0-float(w))*cur_oof+float(w)*cand_oof
                        score=_rmse_np(yv,trial)
                        if score+1e-7<best_score:
                            step=(name,float(w),score,trial)
                            best_score=score
                if step is None:
                    break
                name,w,score,trial_oof=step
                cur_oof=trial_oof.astype(np.float32)
                cur_test=((1.0-w)*cur_test+w*result_dict[name]['test'].astype(np.float32)).astype(np.float32)
                for k in list(weights):
                    weights[k]*=(1.0-w)
                weights[name]=weights.get(name,0.0)+w
                trace.append({'round':rd,'added_model':name,'weight':w,'rmse':score})
            return cur_oof,cur_test,best_score,weights,trace

        hill_oof,hill_test,r_hill,hill_weights,hill_trace=hill_climb_stack(results,y_arr)
        best_single_name=min(results,key=lambda k: results[k]['rmse'])
        best_single_oof=results[best_single_name]['oof']
        best_single_test=results[best_single_name]['test']
        r_best_single=results[best_single_name]['rmse']

        stack_candidates={
            'best_single':(r_best_single,best_single_oof,best_single_test),
            'simple_avg':(r_avg,Sx.mean(1).astype(np.float32),St.mean(1).astype(np.float32)),
            'ridge_stack':(r_stk,oof_s,test_s),
            'hill_stack':(r_hill,hill_oof,hill_test),
        }
        selected_stack_name=min(stack_candidates,key=lambda k: stack_candidates[k][0])
        selected_stack_rmse,final_oof,final_test=stack_candidates[selected_stack_name]

        print(f"\nBest single OOF: {r_best_single:.4f} ({best_single_name})")
        print(f"Simple avg OOF: {r_avg:.4f}")
        print(f"Ridge stk OOF: {r_stk:.4f}  wts={dict(zip(stack_names,wts.round(4)))}")
        print(f"Hill stk OOF: {r_hill:.4f}  wts={ {k:round(v,4) for k,v in hill_weights.items()} }")
        print(f"Selected stack: {selected_stack_name}  OOF={selected_stack_rmse:.4f}")
        # ─ Post-Processing + Submission ───────────────────────────────────
        base=train_df['last_known_tvt'].values.astype(np.float32)
        ytrue=y.values.astype(np.float32)+base
        pf_train=train_df['pf_ancc_delta'].values.astype(np.float32)
        pf_test=test_df['pf_ancc_delta'].values.astype(np.float32)
        dtw_train=train_df['dtw_ens_d'].values.astype(np.float32) if 'dtw_ens_d' in train_df else np.zeros(len(train_df),np.float32)
        dtw_test=test_df['dtw_ens_d'].values.astype(np.float32) if 'dtw_ens_d' in test_df else np.zeros(len(test_df),np.float32)

        def _residual_postprocess(df, model_delta, pf_delta, dtw_delta, alpha, tau, w_pf, w_dtw):
            w_model=max(0.0,1.0-float(w_pf)-float(w_dtw))
            d=(w_model*model_delta.astype(np.float32)+float(w_pf)*pf_delta.astype(np.float32)+float(w_dtw)*dtw_delta.astype(np.float32))
            if tau is not None:
                d=d*(1.-np.exp(-np.maximum(df['md_since'].values.astype(np.float32),0.)/float(tau)))
            return (d*float(alpha)).astype(np.float32)

        def _smooth_values_by_well(df, values, sg_w=0, sg_p=3):
            if not sg_w or sg_w <= 0:
                return values.astype(np.float32)
            out=values.astype(np.float32).copy()
            for well,gp in df.groupby('well',sort=False):
                idx=gp.index.to_numpy()
                v=out[idx]
                n=len(v); wl=min(int(sg_w),n)
                if wl%2==0: wl-=1
                if wl>=int(sg_p)+2:
                    out[idx]=savgol_filter(v,wl,int(sg_p)).astype(np.float32)
            return out

        # Stage 1: choose residual shrinkage, fade-in, and small PF/DTW reference mixing.
        best_cfg=None; best_delta=None; best_r=np.inf
        alpha_grid=np.round(np.arange(0.84,1.061,0.02),2)
        tau_grid=[None,30.,50.,80.,120.,200.,300.]
        w_pf_grid=[0.0,0.03,0.06,0.10]
        w_dtw_grid=[0.0,0.03,0.06,0.10]
        for alpha in alpha_grid:
            for tau in tau_grid:
                for w_pf in w_pf_grid:
                    for w_dtw in w_dtw_grid:
                        if w_pf+w_dtw>0.18:
                            continue
                        d=_residual_postprocess(train_df,final_oof,pf_train,dtw_train,alpha,tau,w_pf,w_dtw)
                        pred=base+d
                        r=root_mean_squared_error(ytrue,pred)
                        if r<best_r:
                            best_r=float(r)
                            best_cfg={'alpha':float(alpha),'tau':tau,'w_pf':float(w_pf),'w_dtw':float(w_dtw),'sg_w':0,'sg_p':0}
                            best_delta=d

        no_smooth_r=float(best_r)

        # Stage 2: tune optional Savitzky-Golay smoothing on absolute OOF predictions.
        best_abs=base+best_delta
        for sg_w in [0,9,13,17,25,35]:
            for sg_p in [2,3]:
                if sg_w and sg_w<=sg_p+1:
                    continue
                cand=_smooth_values_by_well(train_df,best_abs,sg_w,sg_p)
                r=root_mean_squared_error(ytrue,cand)
                if r<best_r:
                    best_r=float(r)
                    best_cfg=dict(best_cfg,sg_w=int(sg_w),sg_p=int(sg_p))
        print(f"Best post-proc: {best_cfg}  abs TVT RMSE={best_r:.4f}")
        ALPHA=best_cfg['alpha']; TAU=best_cfg['tau']; W_PF=best_cfg['w_pf']; W_DTW=best_cfg['w_dtw']; SG_W=best_cfg['sg_w']; SG_P=best_cfg['sg_p']

        sample=pd.read_csv(SAMPLE)
        fb=float(train_df['last_known_tvt'].mean()+train_df['target'].mean())
        test_base=test_df['last_known_tvt'].values.astype(np.float32)

        test_delta_pp=_residual_postprocess(test_df,final_test,pf_test,dtw_test,ALPHA,TAU,W_PF,W_DTW)
        test_pred_abs=test_base+test_delta_pp
        test_pred_smooth=_smooth_values_by_well(test_df,test_pred_abs,SG_W,SG_P)




        candidate_predictions_abs={
            'best_single': test_base + best_single_test,
            'ridge': test_base + test_s,
            'hill': test_base + hill_test,
            'selected_raw': test_base + final_test,
            'no_smooth': test_pred_abs,
            'postproc': test_pred_smooth,
        }
        candidate_oof_rmse={
            'best_single': float(r_best_single),
            'ridge': float(r_stk),
            'hill': float(r_hill),
            'selected_raw': float(selected_stack_rmse),
            'no_smooth': float(no_smooth_r),
            'postproc': float(best_r),
        }

        pf_selector_abs = None
        if bool(globals().get('RUN_TARGET_FREE_SELECTOR_CANDIDATE', True)):
            try:
                selector_sub = _build_target_free_selector_submission(sample)
                pf_selector_abs = selector_sub['tvt'].to_numpy(dtype=np.float32)
                selector_lookup = selector_sub.rename(columns={'tvt': 'pf_selector_tvt'})
                selector_aligned = test_df[['id']].merge(selector_lookup, on='id', how='left')['pf_selector_tvt'].to_numpy(dtype=np.float32)
                if np.isnan(selector_aligned).any():
                    raise RuntimeError('Target-free selector could not align to test feature rows.')
                pf_selector_abs = selector_aligned
                diff_selector = np.abs(test_pred_smooth.astype(float) - pf_selector_abs.astype(float))
                aux_gate = float(globals().get('PF_SELECTOR_AS_AUX_GATED_MAX_WEIGHT', 0.015)) / (
                    1.0 + (diff_selector / float(globals().get('PF_SELECTOR_AS_AUX_GATED_SCALE', 4.0))) ** 2
                )
                postproc_sel15_gated_abs = (1.0 - aux_gate) * test_pred_smooth + aux_gate * pf_selector_abs
                no_smooth_diff_selector = np.abs(test_pred_abs.astype(float) - pf_selector_abs.astype(float))
                no_smooth_aux_gate = float(globals().get('PF_SELECTOR_AS_AUX_GATED_MAX_WEIGHT', 0.015)) / (
                    1.0 + (no_smooth_diff_selector / float(globals().get('PF_SELECTOR_AS_AUX_GATED_SCALE', 4.0))) ** 2
                )
                no_smooth_sel15_gated_abs = (1.0 - no_smooth_aux_gate) * test_pred_abs + no_smooth_aux_gate * pf_selector_abs
                candidate_predictions_abs['pf_selector'] = pf_selector_abs
                candidate_predictions_abs['postproc_sel15_gated'] = postproc_sel15_gated_abs
                candidate_predictions_abs['no_smooth_sel15_gated'] = no_smooth_sel15_gated_abs
                candidate_oof_rmse['pf_selector'] = np.nan
                candidate_oof_rmse['postproc_sel15_gated'] = np.nan
                candidate_oof_rmse['no_smooth_sel15_gated'] = np.nan
                pd.Series({
                    'rows': int(len(selector_sub)),
                    'test_rows_aligned': int(len(pf_selector_abs)),
                    'selector_as_aux_gate_mean': float(np.mean(aux_gate)),
                    'selector_as_aux_gate_p95': float(np.quantile(aux_gate, 0.95)),
                    'selector_as_aux_gate_max': float(np.max(aux_gate)),
                    'mean_abs_stack_diff': float(np.mean(diff_selector)),
                    'p95_abs_stack_diff': float(np.quantile(diff_selector, 0.95)),
                    'mean_abs_no_smooth_diff': float(np.mean(no_smooth_diff_selector)),
                    'p95_abs_no_smooth_diff': float(np.quantile(no_smooth_diff_selector, 0.95)),
                }).to_csv(OUTPUT_DIR / 'target_free_selector_summary.csv')
            except Exception as exc:
                print(f'Target-free PF/beam selector candidate skipped: {exc}')
        aliases={
            'auto':'postproc',
            'auto_oof':'postproc',
            'smooth':'postproc',
            'postprocessed':'postproc',
            'raw':'selected_raw',
            'selected':'selected_raw',
            'hill_stack':'hill',
            'ridge_stack':'ridge',
            'pf':'pf_selector',
            'public_selector':'pf_selector',
            'selector':'pf_selector',
            'sel15_gated':'postproc_sel15_gated',
            'postproc_sel15':'postproc_sel15_gated',
            'no_smooth_sel15':'no_smooth_sel15_gated',
        }
        requested_candidate=str(globals().get('FINAL_V7_CANDIDATE','postproc')).strip().lower()
        selected_candidate=aliases.get(requested_candidate, requested_candidate)
        if selected_candidate not in candidate_predictions_abs:
            raise ValueError(
                f"Unknown FINAL_V7_CANDIDATE={requested_candidate!r}. "
                f"Choose one of {sorted(candidate_predictions_abs)}."
            )

        def _submission_from_prediction(pred_abs):
            frame=pd.DataFrame({'id':test_df['id'].values,'pred':np.asarray(pred_abs,dtype=np.float32)})
            pred_lookup=(frame.groupby('id', as_index=False)['pred'].mean().rename(columns={'pred':'tvt'}))
            cand=sample[['id']].merge(pred_lookup,on='id',how='left')
            missing=int(cand['tvt'].isna().sum())
            cand['tvt']=cand['tvt'].fillna(fb).astype(float)
            if len(cand) != len(sample) or not cand['id'].equals(sample['id']):
                raise RuntimeError('Submission alignment failed for selected v7 candidate.')
            if not np.isfinite(cand['tvt']).all():
                raise RuntimeError('Non-finite TVT values found for selected v7 candidate.')
            return cand[['id','tvt']], missing

        candidate_selection_summary=pd.DataFrame([
            {
                'candidate': name,
                'selected': bool(name == selected_candidate),
                'oof_rmse_used_for_selection': float(candidate_oof_rmse.get(name, np.nan)),
                'tvt_mean': float(np.nanmean(pred)),
                'tvt_std': float(np.nanstd(pred)),
                'tvt_min': float(np.nanmin(pred)),
                'tvt_max': float(np.nanmax(pred)),
            }
            for name, pred in candidate_predictions_abs.items()
        ]).sort_values(['selected','oof_rmse_used_for_selection'], ascending=[False, True])

        sub, missing_predictions = _submission_from_prediction(candidate_predictions_abs[selected_candidate])
        sub.to_csv(OUT,index=False)
        globals()['FINAL_BASE_SOURCE_LABEL'] = selected_candidate

        print(f"\n✅  {OUT}  {len(sub)} rows")
        print("\n─── Final Summary ───────────────────────────")
        for k,v in results.items(): print(f"  {k}: OOF residual RMSE = {v['rmse']:.4f}")
        print(f"  Ridge stk: {r_stk:.4f}  |  Hill stk: {r_hill:.4f}  |  Selected: {selected_stack_name}  |  PostProc: {best_r:.4f}")
        print(f"  Final candidate: {selected_candidate}  (requested: {requested_candidate})  OOF proxy={candidate_oof_rmse[selected_candidate]:.4f}")
        print(sub.head(8).to_string(index=False))

        # Reports for prediction diagnostics and submission contract tracking.
        model_summary = pd.DataFrame(
            [{'model': k, 'metric_space': 'residual_delta', 'oof_rmse': float(v['rmse']), 'selected_stack': selected_stack_name} for k, v in results.items()]
            + [
                {'model': 'best_single', 'metric_space': 'residual_delta', 'oof_rmse': float(r_best_single), 'selected_stack': selected_stack_name},
                {'model': 'simple_avg', 'metric_space': 'residual_delta', 'oof_rmse': float(r_avg), 'selected_stack': selected_stack_name},
                {'model': 'ridge_stack', 'metric_space': 'residual_delta', 'oof_rmse': float(r_stk), 'selected_stack': selected_stack_name},
                {'model': 'hill_stack', 'metric_space': 'residual_delta', 'oof_rmse': float(r_hill), 'selected_stack': selected_stack_name},
                {'model': 'postprocessed_abs_tvt', 'metric_space': 'absolute_tvt', 'oof_rmse': float(best_r), 'selected_stack': selected_stack_name},
            ]
        )
        model_summary.to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_model_summary.csv', index=False)
        pd.DataFrame([{'model': k, 'ridge_weight': float(w)} for k, w in zip(stack_names, wts)]).to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_ridge_weights.csv', index=False)
        pd.DataFrame([{'model': k, 'hill_weight': float(v)} for k, v in hill_weights.items()]).to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_hill_weights.csv', index=False)
        pd.DataFrame(hill_trace).to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_hill_trace.csv', index=False)
        pd.DataFrame(fold_rows).to_csv(OUTPUT_DIR / 'v7_dtw_super_stack_fold_report.csv', index=False)
        candidate_selection_summary.to_csv(OUTPUT_DIR / 'v7_candidate_selection_summary.csv', index=False)
        display(candidate_selection_summary)
        contract_guard = pd.DataFrame([{
            'final_source': str(SUPER_STACK_SUBMISSION_OUTPUT),
            'final_output': str(OUT),
            'feature_count': int(len(feature_cols)),
            'train_rows': int(len(train_df)),
            'test_rows': int(len(test_df)),
            'best_single_oof_rmse': float(r_best_single),
            'simple_avg_oof_rmse': float(r_avg),
            'ridge_stack_oof_rmse': float(r_stk),
            'hill_stack_oof_rmse': float(r_hill),
            'selected_stack_oof_rmse': float(selected_stack_rmse),
            'postprocessed_abs_tvt_oof_rmse': float(best_r),
            'postprocess_alpha': float(ALPHA),
            'postprocess_tau': np.nan if TAU is None else float(TAU),
            'postprocess_w_pf': float(W_PF),
            'postprocess_w_dtw': float(W_DTW),
            'postprocess_sg_window': int(SG_W),
            'postprocess_sg_poly': int(SG_P),
            'model_count': int(len(results)),
            'ridge_weights_json': json.dumps({k: float(w) for k, w in zip(stack_names, wts)}, sort_keys=True),
            'ridge_weights_raw_json': json.dumps({k: float(w) for k, w in zip(stack_names, ridge.coef_)}, sort_keys=True),
            'hill_weights_json': json.dumps({k: float(v) for k, v in hill_weights.items()}, sort_keys=True),
            'selected_stack': selected_stack_name,
            'final_candidate_requested': requested_candidate,
            'final_candidate_selected': selected_candidate,
            'final_candidate_oof_rmse': float(candidate_oof_rmse[selected_candidate]),
            'train_matrix_memory_mb': float(train_matrix_mb),
            'test_matrix_memory_mb': float(test_matrix_mb),
            'formation_count': int(len(FORMATIONS)),
            'beam_count': int(len(BEAMS)),
            'dtw_enabled': True,
            'dtw_radii_json': json.dumps([int(r) for r in DTW_RADII]),
            'dtw_stoch_k': int(DTW_STOCH_K),
            'dtw_stride': int(DTW_STRIDE),
            'feature_build_ncpu': int(NCPU),
            'dtw_anchor_abs_error_train_median': float(train_df['dtw_anchor_abs_error'].median()) if 'dtw_anchor_abs_error' in train_df else np.nan,
            'dtw_anchor_abs_error_test_median': float(test_df['dtw_anchor_abs_error'].median()) if 'dtw_anchor_abs_error' in test_df else np.nan,
            'selfcorr_enabled': True,
            'pf_tvt_z_enabled': True,
            'pf_ancc_enabled': True,
            'dense_ancc_enabled': True,
            'formation_train_exclude_self': True,
            'formation_test_exclude_self': False,
            'missing_predictions_filled': int(missing_predictions),
            'submission_rows': int(len(sub)),
            'submission_tvt_mean': float(sub['tvt'].mean()),
            'submission_tvt_std': float(sub['tvt'].std()),
            'submission_tvt_min': float(sub['tvt'].min()),
            'submission_tvt_max': float(sub['tvt'].max()),
        }])
        contract_guard.to_csv(OUTPUT_DIR / 'submission_contract_guard_summary_v7.csv', index=False)


Super-stack final solution is skipped for ridge_artifact_experiment profile.


## Ridge Artifact Engine + Physical/PF Heuristic Profile

The ridge artifact engine writes:

$$
T_i^{\mathrm{blend}} = w_r T_i^{\mathrm{ridge}} + (1-w_r)T_i^{\mathrm{heur}}.
$$

The current `ridge_artifact_experiment` settings reproduce the sp45 projection parameter set: $w_r=0.30$, $N_p=500$, $S=128$, $\sigma_0=4.5$, and a robust degree-5 projection in $U=T+Z-A_w$ space. The same profile can be used for nearby probes by changing $w_r$, $N_p$, $S$, $\sigma_0$, or $d$ in the first code cell. The older `ridge_artifact_exact` preset keeps the non-projected ridge-artifact reference values.


In [6]:
# Artifact-backed ridge / heuristic profile.
if not bool(globals().get('RUN_RIDGE_ARTIFACT_PROFILE', False)):
    print('Ridge artifact profile skipped.')
else:
    from lightgbm import LGBMRegressor, log_evaluation, early_stopping
    from sklearn.metrics import root_mean_squared_error
    from sklearn.model_selection import GroupKFold
    from sklearn.linear_model import Ridge
    from catboost import CatBoostRegressor
    from scipy.spatial import cKDTree
    from scipy.signal import savgol_filter
    from joblib import Parallel, delayed
    try:
        from koolbox import Trainer
    except ModuleNotFoundError as exc:
        raise RuntimeError('The ridge artifact profiles require the original notebook dependency: koolbox.') from exc
    from pathlib import Path
    from numba import njit
    import matplotlib.pyplot as plt
    import multiprocessing
    import seaborn as sns
    import pandas as pd
    import numpy as np
    import warnings
    import joblib
    import time
    import glob
    import os

    warnings.filterwarnings("ignore")


    class CFG:
        dataset_path = next(
            (Path(path) for path in COMPETITION_DATA_ROOTS if Path(path).exists()),
            Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
        )
        artifacts_path = next(
            (Path(path) for path in RIDGE_ARTIFACT_ROOTS if Path(path).exists()),
            Path('/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts'),
        )

        seed = 42
        n_splits = 5
        cv = GroupKFold(n_splits=n_splits)
        metric = root_mean_squared_error


    SELECTOR_N_EVAL_THRESHOLD = 4840.0
    SELECTOR_Z_SPAN_THRESHOLDS = (136.73000000000016, 185.5133333333342)

    SELECTOR_BIN_VARIANTS = {
        0: 'pf_scale_5_hold_0.2',
        1: 'pf_scale_3_hold_0.15',
        2: 'pf_scale_12_beam_0.2_hold_0.15',
        3: 'pf_scale_5_hold_0.15',
        4: 'pf_scale_5_beam_0.05_hold_0.05',
        5: 'pf_scale_12_beam_0.2_hold_0.05',
    }

    SELECTOR_GLOBAL_VARIANT = 'pf_scale_8_hold_0.2'
    SELECTOR_SCALES = (3.0, 5.0, 8.0, 12.0)

    FORMATION_COLS = ['ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA']

    BEAM_CONFIGS = [
        (10, 20.0, 144.0, 2),
        (10,  8.0,  64.0, 2),
        ( 8, 35.0, 220.0, 1),
        (10, 14.0,  90.0, 5),
        (20,  4.0,  36.0, 3),
        (12, 12.0, 100.0, 3),
        (15, 25.0, 180.0, 2),
        (20, 30.0, 200.0, 2),
        (15, 10.0,  80.0, 4),
        (25,  6.0,  50.0, 3),
        (10, 40.0, 300.0, 1),
        (12, 18.0, 120.0, 5),
        (30,  8.0,  70.0, 2),
        (10, 50.0, 400.0, 0),
    ]


    def tvt_from_contacts(hw_tr, tw_tr, ref_col='EGFDU'):
        tw_g = tw_tr.dropna(subset=['Geology'])
        ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
        if np.isnan(ref_tvt):
            ref_col = tw_g['Geology'].iloc[0]
            ref_tvt = tw_g[tw_g['Geology'] == ref_col]['TVT'].min()
        offset = (hw_tr['TVT'] - (ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]))).mean()
        return ref_tvt - (hw_tr['Z'] - hw_tr[ref_col]) + offset


    def load_well(wid, split='train'):
        base = CFG.dataset_path / split
        hw = pd.read_csv(base / f'{wid}__horizontal_well.csv')
        tw = pd.read_csv(base / f'{wid}__typewell.csv')
        return hw, tw


    def run_particle_filter(hw, tw, n_particles=500, seed=42):
        tw_s   = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(ev) == 0:
            return hw['TVT_input'].values.astype(float).copy(), 0.0

        last     = kn.iloc[-1]
        last_tvt = float(last['TVT_input'])
        last_Z   = float(last['Z'])
        last_MD  = float(last['MD'])

        tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
        gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

        tail = kn.tail(30)
        dt = np.diff(tail['TVT_input'].values)
        dz = np.diff(tail['Z'].values)
        dm = np.diff(tail['MD'].values)
        m  = dm > 0
        ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

        N   = n_particles
        rng = np.random.default_rng(seed)
        ls   = last_tvt + last_Z
        init_spread = float(globals().get('RIDGE_ARTIFACT_PF_INIT_SPREAD', 2.0))
        pos  = ls + init_spread * rng.standard_normal(N)
        rate = ir + 0.01 * rng.standard_normal(N)
        w    = np.ones(N) / N

        MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5

        md_v = ev['MD'].values.astype(float)
        z_v  = ev['Z'].values.astype(float)
        # Interpolate GR gaps before tracking
        gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
        gr_v = gr_interp.values.astype(float)[ev.index]

        out_vals = hw['TVT_input'].values.astype(float).copy()
        res = np.empty(len(ev))
        prev_MD = last_MD
        log_lik = 0.0

        for i in range(len(ev)):
            dm_step = max(md_v[i] - prev_MD, 1.0)
            rate = MOM * rate + VN * rng.standard_normal(N)
            pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
            tvt_p = pos - z_v[i]
            tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
            pos   = tvt_p + z_v[i]

            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d  = (gr_v[i] - eg) / gs
            lk = np.exp(-0.5 * np.minimum(d**2, 600.))
            lk = np.maximum(lk, 1e-300)
            avg_lk = float((w * lk).sum())
            log_lik += np.log(max(avg_lk, 1e-300))
            w = w * lk
            ws = w.sum()
            w = w / ws if ws > 0 else np.ones(N) / N

            n_eff = 1.0 / (w**2).sum()
            if n_eff < RESAMP * N:
                cum = np.cumsum(w)
                u0  = rng.uniform(0, 1.0 / N)
                idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
                pos  = pos[idx]  + RP * rng.standard_normal(N)
                rate = rate[idx] + RR * rng.standard_normal(N)
                w    = np.ones(N) / N

            res[i] = float(np.dot(w, pos - z_v[i]))
            prev_MD = md_v[i]

        out_vals[list(ev.index)] = res
        return out_vals, log_lik


    def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
        preds = []
        liks  = []
        for s in range(n_seeds):
            p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
            preds.append(p)
            liks.append(ll)

        liks   = np.array(liks)
        liks_n = liks - liks.max()
        weights = np.exp(liks_n / scale)
        weights /= weights.sum()

        return (weights[:, None] * np.stack(preds, 0)).sum(0)


    def run_pf_lik_ensemble_scales(hw, tw, scales=SELECTOR_SCALES, n_particles=500, n_seeds=128):
        preds = []
        liks = []
        for s in range(n_seeds):
            p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
            preds.append(p)
            liks.append(ll)
        pred_arr = np.stack(preds, 0)
        liks = np.array(liks)
        liks_n = liks - liks.max()
        out = {}
        for scale in scales:
            weights = np.exp(liks_n / float(scale))
            weights /= weights.sum()
            out[f'pf_scale_{scale:g}'] = (weights[:, None] * pred_arr).sum(0)
        out['pf_mean'] = pred_arr.mean(0)
        return out


    def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
        n  = len(hgr)
        nt = len(tw_tvt)
        if n == 0:
            return np.array([last_tvt])

        if r > 0 and n > max(3, 2 * r + 1):
            win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()

        si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

        MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
        MC    = mc * np.array([2., 1., 0., 1., 2.])

        bidx  = np.full(bs, si, dtype=np.int64)
        bcost = np.full(bs, np.inf)
        bcost[0] = 0.
        bn = 1

        result = np.zeros(n)

        for step in range(n):
            gv = sgr[step]
            ni = bidx[:bn, None] + MOVES[None, :]
            ci = np.clip(ni, 0, nt - 1)
            valid = (ni >= 0) & (ni < nt)

            gr_e = (gv - tw_gr[ci])**2 / es
            tot  = bcost[:bn, None] + gr_e + MC[None, :]
            tot  = np.where(valid, tot, np.inf)

            ni_f  = ni.flatten()
            tot_f = tot.flatten()
            vf    = valid.flatten()
            ni_f  = ni_f[vf]
            tot_f = tot_f[vf]

            order = np.argsort(tot_f)
            ni_s  = ni_f[order]
            tot_s = tot_f[order]

            _, first = np.unique(ni_s, return_index=True)
            ni_u  = ni_s[first]
            tot_u = tot_s[first]

            kept = min(bs, len(ni_u))
            top  = np.argpartition(tot_u, min(kept - 1, len(tot_u) - 1))[:kept]
            top  = top[np.argsort(tot_u[top])]

            bidx[:kept]  = ni_u[top]
            bcost[:kept] = tot_u[top]
            if kept < bs:
                bidx[kept:]  = bidx[kept - 1]
                bcost[kept:] = np.inf
            bn = kept

            result[step] = tw_tvt[bidx[0]]

        return result


    def run_beam_ensemble(hw, tw):
        kn = hw[hw['TVT_input'].notna()]
        ev = hw[hw['TVT_input'].isna()]
        if len(ev) == 0:
            return hw['TVT_input'].values.astype(float).copy()

        last_tvt = float(kn.iloc[-1]['TVT_input'])
        tw_s  = tw.sort_values('TVT')
        tw_tvt = tw_s['TVT'].values.astype(float)
        tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

        gr_all = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean()).values.astype(float)
        hgr    = gr_all[ev.index]

        beam_results = [beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
                        for (bs, mc, es, r) in BEAM_CONFIGS]

        beam_mean = np.stack(beam_results, 0).mean(0)

        out = hw['TVT_input'].values.astype(float).copy()
        out[list(ev.index)] = beam_mean
        return out


    def selector_well_code(hw):
        eval_mask = hw['TVT_input'].isna().to_numpy()
        n_eval = float(eval_mask.sum())
        z_eval = hw.loc[eval_mask, 'Z'].values.astype(float)
        z_span = float(np.nanmax(z_eval) - np.nanmin(z_eval)) if len(z_eval) else 0.0
        n_bin = int(n_eval > SELECTOR_N_EVAL_THRESHOLD)
        z_bin = int(np.searchsorted(SELECTOR_Z_SPAN_THRESHOLDS, z_span, side='right'))
        code = n_bin + 2 * z_bin
        variant = SELECTOR_BIN_VARIANTS.get(code, SELECTOR_GLOBAL_VARIANT)
        return code, variant, n_eval, z_span


    def parse_selector_variant(name):
        parts = name.split('_')
        scale = float(parts[2])
        beam_weight = 0.0
        hold_weight = 0.0
        if 'beam' in parts:
            beam_weight = float(parts[parts.index('beam') + 1])
        if 'hold' in parts:
            hold_weight = float(parts[parts.index('hold') + 1])
        return scale, beam_weight, hold_weight


    def apply_selector_variant(name, pf_by_scale, tvt_beam, last_known_tvt):
        scale, beam_weight, hold_weight = parse_selector_variant(name)
        base = pf_by_scale.get(f'pf_scale_{scale:g}')
        if base is None:
            base = pf_by_scale[SELECTOR_GLOBAL_VARIANT.split('_beam_')[0].split('_hold_')[0]]
        pred = (1.0 - beam_weight) * base + beam_weight * tvt_beam
        pred = (1.0 - hold_weight) * pred + hold_weight * last_known_tvt
        return pred

    SEED=42
    NCPU=min(4,multiprocessing.cpu_count())

    FORMATIONS=["ANCC","ASTNU","ASTNL","EGFDU","EGFDL","BUDA"]
    PLANE_K=10; DENSE_SPW=60; DENSE_K=20; N_SPLITS=5

    BEAMS=[
        (10,20.0,144.0,2,"cons"),
        (10, 8.0, 64.0,2,"loose"),
        ( 8,35.0,220.0,1,"vcons"),
        (10,14.0, 90.0,5,"sm5"),
        (20, 4.0, 36.0,3,"vloose"),
        (12,12.0,100.0,3,"mid"),
        (15,25.0,180.0,2,"stiff"),
    ]

    PF_N=600; ANCC_N=600
    PF_MOM=0.993; PF_VN=0.005; PF_PN=0.01
    PF_GR_SIG_MIN=10.; PF_GR_SIG_MAX=60.; PF_GR_SIG_DEF=30.
    PF_INIT_V_STD=0.02; PF_INIT_SPR=0.5; PF_RESAMP=0.5
    PF_ROUGH_P=0.2; PF_ROUGH_V=0.003; PF_GR_WIN=5; PF_GR_WT=0.3
    ANCC_ALPHA=0.998; ANCC_RN=0.002; ANCC_PN=0.005
    ANCC_IR=0.01; ANCC_IS=0.3; ANCC_RP=0.1; ANCC_RR=0.001

    @njit(cache=True)
    def _interp1(grid, v, vmin, step):
        i = int((v - vmin) / step)
        if i < 0: return grid[0]
        n = len(grid) - 1
        if i >= n: return grid[n]
        t = (v - vmin) / step - i
        return grid[i]*(1.-t) + grid[i+1]*t

    @njit(cache=True)
    def _resamp(pos, aux, w, N, rp, rv):
        cum = np.zeros(N+1)
        for j in range(N): cum[j+1]=cum[j]+w[j]
        u0=np.random.uniform(0.,1./N)
        np2=np.empty(N); na=np.empty(N); ci=0
        for j in range(N):
            u=u0+j/N
            while ci<N-1 and cum[ci+1]<u: ci+=1
            np2[j]=pos[ci]+rp*np.random.randn()
            na[j] =aux[ci]+rv*np.random.randn()
        return np2,na

    @njit(cache=True)
    def _beam_jit(sgr, tw_gr, si, BS, mc, es):
        """Beam search ±2 delta, Numba JIT."""
        n=len(sgr); nt=len(tw_gr); MAX=BS*6
        bidx=np.zeros(BS,np.int64); bidx[0]=si
        bcost=np.full(BS,1e30);     bcost[0]=0.; bn=np.int64(1)
        hI=np.zeros((n,BS),np.int64); hP=np.zeros((n,BS),np.int64)
        cI=np.zeros(MAX,np.int64); cC=np.full(MAX,1e30); cP=np.zeros(MAX,np.int64)
        for step in range(n):
            gv=sgr[step]; nc=np.int64(0)
            for bi in range(bn):
                idx=bidx[bi]; cost=bcost[bi]
                for d in range(-2,3):            # ±2: TVT can go down
                    ni=idx+d
                    if ni<0 or ni>=nt: continue
                    tot=cost+(gv-tw_gr[ni])**2/es+mc*(d if d>=0 else -d)
                    fnd=np.int64(-1)
                    for ci in range(nc):
                        if cI[ci]==ni: fnd=ci; break
                    if fnd>=0:
                        if tot<cC[fnd]: cC[fnd]=tot; cP[fnd]=bi
                    else:
                        if nc<MAX: cI[nc]=ni; cC[nc]=tot; cP[nc]=bi; nc+=1
            kept=min(BS,nc)
            for i in range(kept):
                mi=i
                for j in range(i+1,nc):
                    if cC[j]<cC[mi]: mi=j
                if mi!=i:
                    cI[i],cI[mi]=cI[mi],cI[i]
                    cC[i],cC[mi]=cC[mi],cC[i]
                    cP[i],cP[mi]=cP[mi],cP[i]
            hI[step,:kept]=cI[:kept]; hP[step,:kept]=cP[:kept]
            bidx[:kept]=cI[:kept]; bcost[:kept]=cC[:kept]; bn=kept
        best=np.int64(0)
        for b in range(1,bn):
            if bcost[b]<bcost[best]: best=b
        path=np.zeros(n,np.int64); b=best
        for s in range(n-1,-1,-1): path[s]=hI[s,b]; b=hP[s,b]
        return path

    @njit(cache=True)
    def _pf_ancc(md_v,z_v,gr_v,gg,vmin,step,gs,ls,ir,N,
                  ALPHA,RN,PN,IS,RP,RR,RESAMP):
        pos=np.empty(N); rate=np.empty(N); w=np.ones(N)/N
        for j in range(N):
            pos[j]=ls+IS*np.random.randn()
            rate[j]=ir+0.01*np.random.randn()
        pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.
        for i in range(len(md_v)):
            dm=md_v[i]-pm; dm=max(dm,1.)
            for j in range(N):
                rate[j]=ALPHA*rate[j]+RN*np.random.randn()
                pos[j]+=rate[j]*dm+PN*np.random.randn()
                tvt_j=pos[j]-z_v[i]
                tvt_j=max(tvt_j,vmin-50.); tvt_j=min(tvt_j,vmin+len(gg)*step+50.)
                pos[j]=tvt_j+z_v[i]
            if not np.isnan(gr_v[i]):
                ws=0.
                for j in range(N):
                    eg=_interp1(gg,pos[j]-z_v[i],vmin,step)
                    d=(gr_v[i]-eg)/gs
                    lk=max(np.exp(-0.5*d*d) if d*d<600. else 0.,1e-300)
                    w[j]*=lk; ws+=w[j]
                if ws>0.:
                    for j in range(N): w[j]/=ws
                else:
                    for j in range(N): w[j]=1./N
            ne=0.
            for j in range(N): ne+=w[j]*w[j]
            if 1./ne<RESAMP*N:
                pos,rate=_resamp(pos,rate,w,N,RP,RR)
                for j in range(N): w[j]=1./N
            tv=0.
            for j in range(N): tv+=w[j]*(pos[j]-z_v[i])
            pts[i]=tv; va=0.
            for j in range(N): va+=w[j]*(pos[j]-z_v[i]-tv)**2
            std_[i]=va**0.5; pm=md_v[i]
        return pts,std_

    @njit(cache=True)
    def _pf_z(md_v,z_v,gr_v,gr_sm_v,gg_p,gg_s,vmin,step,
              gs,ip,iv,beta,icpt,zsig,N,
              MOM,VN,PN,GR_WT,RP,RV,RESAMP):
        pos=np.empty(N); vel=np.empty(N); w=np.ones(N)/N
        for j in range(N):
            pos[j]=ip+0.5*np.random.randn()
            vel[j]=iv+0.02*np.random.randn()
        pts=np.empty(len(md_v)); std_=np.empty(len(md_v)); pm=md_v[0]-1.; pz=z_v[0]-1.
        for i in range(len(md_v)):
            dm=md_v[i]-pm; dm=max(dm,1.)
            dzd=(z_v[i]-pz)/dm; ve=beta*dzd+icpt
            for j in range(N):
                vel[j]=MOM*vel[j]+VN*np.random.randn()
                pos[j]+=vel[j]*dm+PN*np.random.randn()
                pos[j]=max(pos[j],vmin-50.); pos[j]=min(pos[j],vmin+len(gg_p)*step+50.)
            if not np.isnan(gr_v[i]):
                ws=0.
                for j in range(N):
                    ep=_interp1(gg_p,pos[j],vmin,step)
                    dp=(gr_v[i]-ep)/gs
                    lp=max(np.exp(-0.5*dp*dp) if dp*dp<600. else 0.,1e-300)
                    if not np.isnan(gr_sm_v[i]):
                        es=_interp1(gg_s,pos[j],vmin,step)
                        ds=(gr_sm_v[i]-es)/(gs*1.5)
                        ls=max(np.exp(-0.5*ds*ds) if ds*ds<600. else 0.,1e-300)
                        lk=(1.-GR_WT)*lp+GR_WT*ls
                    else: lk=lp
                    lk=max(lk,1e-300); w[j]*=lk; ws+=w[j]
                if ws>0.:
                    for j in range(N): w[j]/=ws
                else:
                    for j in range(N): w[j]=1./N
            ws2=0.
            for j in range(N):
                dv=(vel[j]-ve)/max(zsig*2.,0.005)
                lz=max(np.exp(-0.5*dv*dv) if dv*dv<600. else 0.,1e-300)
                w[j]*=lz; ws2+=w[j]
            if ws2>0.:
                for j in range(N): w[j]/=ws2
            else:
                for j in range(N): w[j]=1./N
            ne=0.
            for j in range(N): ne+=w[j]*w[j]
            if 1./ne<RESAMP*N:
                pos,vel=_resamp(pos,vel,w,N,RP,RV)
                for j in range(N): w[j]=1./N
            wm=0.
            for j in range(N): wm+=w[j]*pos[j]
            pts[i]=wm; va=0.
            for j in range(N): va+=w[j]*(pos[j]-wm)**2
            std_[i]=va**0.5; pm=md_v[i]; pz=z_v[i]
        return pts,std_

    # Dense grid for O(1) typewell lookup
    def _grid(tw_tvt,tw_gr,step=0.2):
        tmin=float(tw_tvt.min()); tmax=float(tw_tvt.max())
        tvt_g=np.arange(tmin,tmax+step,step)
        return np.interp(tvt_g,tw_tvt,tw_gr).astype(np.float64),float(tmin),float(step)

    def _gr_sig(hw,tw_tvt,tw_gr):
        kn=hw[hw['TVT_input'].notna()&hw['GR'].notna()]
        if len(kn)<20: return float(PF_GR_SIG_DEF)
        return float(np.clip(np.std(kn['GR'].values-np.interp(kn['TVT_input'].values,tw_tvt,tw_gr)),
                              PF_GR_SIG_MIN,PF_GR_SIG_MAX))

    def _nn(arr,v):
        i=int(np.searchsorted(arr,v,'left'))
        if i>=len(arr): return len(arr)-1
        if i>0 and abs(arr[i-1]-v)<=abs(arr[i]-v): return i-1
        return i

    def _smooth(vals,fb,r):
        s=pd.Series(vals,dtype='float32').interpolate(limit_direction='both').fillna(fb)
        return (s.rolling(r*2+1,center=True,min_periods=1).mean() if r>0 else s).to_numpy(np.float32)

    def beam_search(gr_h,tw_tvt,tw_gr,start_tvt,bs,mc,es,r):
        si=_nn(tw_tvt,start_tvt)
        sgr=_smooth(gr_h,float(np.nanmean(tw_gr)),r).astype(np.float64)
        path=_beam_jit(sgr,tw_gr.astype(np.float64),si,bs,float(mc),float(es))
        return tw_tvt[path].astype(np.float32)

    def run_pf_ancc(hw,tw_tvt,tw_gr,N=ANCC_N):
        gs=_gr_sig(hw,tw_tvt,tw_gr)
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        ls=float(kn['TVT_input'].iloc[-1]+kn['Z'].iloc[-1])
        tail=kn.tail(30); dt=np.diff(tail['TVT_input'].values)
        dz=np.diff(tail['Z'].values); dm=np.diff(tail['MD'].values); m=dm>0
        ir=float(np.median((dt+dz)[m]/dm[m])) if m.sum()>=3 else 0.
        gg,gmin,gst=_grid(tw_tvt,tw_gr)
        pts,std=_pf_ancc(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                          ev['GR'].values.astype(np.float64),gg,gmin,gst,
                          gs,ls,ir,N,ANCC_ALPHA,ANCC_RN,ANCC_PN,ANCC_IS,ANCC_RP,ANCC_RR,PF_RESAMP)
        return pts.astype(np.float32),std.astype(np.float32)

    def run_pf_z(hw,tw_tvt,tw_gr,N=PF_N):
        gs=_gr_sig(hw,tw_tvt,tw_gr)
        tw_s=pd.Series(tw_gr).rolling(PF_GR_WIN,center=True,min_periods=1).mean().values.astype(np.float32)
        kna=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0: return np.array([]),np.array([])
        dz_k=np.diff(kna['Z'].values); dvt=np.diff(kna['TVT_input'].values)
        dmd_k=np.diff(kna['MD'].values); m2=dmd_k>0
        if m2.sum()>=10:
            vz=dz_k[m2]/dmd_k[m2]; vt=dvt[m2]/dmd_k[m2]
            A=np.column_stack([vz,np.ones_like(vz)]); c,_,_,_=np.linalg.lstsq(A,vt,rcond=None)
            beta,icpt,zsig=float(c[0]),float(c[1]),max(float(np.std(vt-(c[0]*vz+c[1]))),0.001)
        else: beta,icpt,zsig=-1.,0.,0.1
        t2=kna.tail(20); dvt2=np.diff(t2['TVT_input'].values); dmd2=np.diff(t2['MD'].values); m3=dmd2>0
        iv=float(np.median(dvt2[m3]/dmd2[m3])) if m3.sum()>=3 else 0.
        gg,gmin,gst=_grid(tw_tvt,tw_gr)
        gs2,_,_=_grid(tw_tvt,tw_s)
        gr_sm=hw['GR'].rolling(PF_GR_WIN,center=True,min_periods=1).mean()
        pts,std=_pf_z(ev['MD'].values.astype(np.float64),ev['Z'].values.astype(np.float64),
                       ev['GR'].values.astype(np.float64),
                       gr_sm.loc[ev.index].values.astype(np.float64),
                       gg,gs2,gmin,gst,gs,float(kna['TVT_input'].iloc[-1]),iv,
                       beta,icpt,zsig,N,
                       PF_MOM,PF_VN,PF_PN,PF_GR_WT,PF_ROUGH_P,PF_ROUGH_V,PF_RESAMP)
        return pts.astype(np.float32),std.astype(np.float32)


    _md=np.linspace(1,50,20,np.float64); _z=np.zeros(20,np.float64); _gr=np.full(20,50.,np.float64)
    _gg=np.linspace(45,55,100,np.float64)
    _pf_ancc(_md,_z,_gr,_gg,45.,0.1,20.,50.,0.,8,0.998,0.002,0.005,0.3,0.1,0.001,0.5)
    _pf_z(_md,_z,_gr,_gr,_gg,_gg,45.,0.1,20.,50.,0.,-1.,0.,0.1,8,0.993,0.005,0.01,0.3,0.2,0.003,0.5)
    _beam_jit(np.random.randn(30),np.random.randn(50),25,8,15.,100.)

    def robust_slope(x,y,w=None):
        x=np.asarray(x,float); y=np.asarray(y,float)
        m=np.isfinite(x)&np.isfinite(y)
        if m.sum()<2 or np.std(x[m])<1e-6: return 0.
        return float(np.polyfit(x[m],y[m],1)[0])

    def affine_cal(kgr,tw_at_k,min_pts=20):
        v=np.isfinite(kgr)&np.isfinite(tw_at_k)
        if v.sum()<min_pts or np.std(tw_at_k[v])<1e-6:
            return 1.,float(np.nanmean(kgr)-np.nanmean(tw_at_k)) if v.any() else 0.
        a,b=np.polyfit(tw_at_k[v],kgr[v],1); return float(a),float(b)

    def seg_b_well(ktvt,kz,form_col):
        """Segment b_well: early/mid/late thirds + full prefix.
        Returns (b_full, b_early, b_mid, b_late, b_wls) for feature richness."""
        bv=ktvt+kz-form_col; n=len(bv)
        b_full=float(np.median(bv))
        b_late=float(np.median(bv[max(0,n-50):])) if n>=5 else b_full
        t1,t2=n//3, 2*n//3
        b_early=float(np.median(bv[:max(1,t1)])) if t1>0 else b_full
        b_mid  =float(np.median(bv[t1:max(t1+1,t2)])) if t2>t1 else b_full
        # WLS (tail-upweighted)
        w=np.exp(0.02*np.arange(n)); w/=w.sum()
        b_wls=float(np.dot(w,bv))
        return b_full,b_early,b_mid,b_late,b_wls

    def multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3):
        """Multi-scale NCC. Returns score-weighted ensemble + per-scale signals."""
        out=[]
        for hw in hws:
            win=2*hw+1; nk=len(kgr); nh=len(hgr)
            if nk<win+1 or nh==0:
                out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
            kg=pd.Series(kgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
            hg=pd.Series(hgr).rolling(5,center=True,min_periods=1).mean().values.astype(np.float32)
            sts=np.arange(0,nk-win+1,stride,dtype=np.int32); M=len(sts)
            if M==0:
                out.append((np.full(nh,ktvt[-1],np.float32),np.zeros(nh,np.float32))); continue
            C=kg[sts[:,None]+np.arange(win,dtype=np.int32)[None,:]].astype(np.float32)
            Cn=(C-C.mean(1,keepdims=True))/(C.std(1,keepdims=True)+1e-6)
            hp=np.pad(hg,hw,mode='edge')
            H=hp[np.arange(nh)[:,None]+np.arange(win)[None,:]].astype(np.float32)
            Hn=(H-H.mean(1,keepdims=True))/(H.std(1,keepdims=True)+1e-6)
            ncc=Hn@Cn.T/win; best=ncc.argmax(1); score=ncc.max(1).astype(np.float32)
            out.append((ktvt[np.clip(sts[best]+hw,0,nk-1)].astype(np.float32),score))
        # Score-weighted ensemble (NEW: softmax-weighted combination)
        tvts=np.stack([o[0] for o in out],1); scores=np.stack([o[1] for o in out],1)
        sw=np.exp(3.*scores); sw/=sw.sum(1,keepdims=True)+1e-9
        sc_ens=(tvts*sw).sum(1).astype(np.float32)
        return out, sc_ens   # [(tvt8,sc8),(tvt15,sc15),(tvt25,sc25)], ensemble

    class FormationPlaneKNN:
        def __init__(self,well_ids,data_dir):
            rows=[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y']+FORMATIONS).dropna()
                except: continue
                if len(df)==0: continue
                row={'wid':wid,'x':float(df['X'].median()),'y':float(df['Y'].median())}
                for c in FORMATIONS: row[f'{c}_m']=float(df[c].median())
                rows.append(row)
            self.df=pd.DataFrame(rows); self.wmap={w:i for i,w in enumerate(self.df['wid'])}
            xy=self.df[['x','y']].to_numpy(); self.scale=np.where(xy.std(0)<1e-3,1.,xy.std(0))
            self.tree=cKDTree(xy/self.scale)
            self.xa=self.df['x'].to_numpy(); self.ya=self.df['y'].to_numpy()
            self.fa=self.df[[f'{c}_m' for c in FORMATIONS]].to_numpy(np.float64)

        def impute(self,xy_q,self_wid=None,k=PLANE_K):
            q=xy_q/self.scale; nf=min(k+5,len(self.df))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid in self.wmap: dist=np.where(idx==self.wmap[self_wid],np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.).astype(np.float64)
            xn=self.xa[ik]; yn=self.ya[ik]; fn=self.fa[ik]; wx=w*xn; wy=w*yn
            A=np.zeros((len(q),3,3))
            A[:,0,0]=(wx*xn).sum(1); A[:,0,1]=(wx*yn).sum(1); A[:,0,2]=wx.sum(1)
            A[:,1,0]=A[:,0,1]; A[:,1,1]=(wy*yn).sum(1); A[:,1,2]=wy.sum(1)
            A[:,2,0]=A[:,0,2]; A[:,2,1]=A[:,1,2]; A[:,2,2]=w.sum(1)
            A[:,0,0]+=1e-9; A[:,1,1]+=1e-9; A[:,2,2]+=1e-9
            rhs=np.stack([(wx[:,:,None]*fn).sum(1),(wy[:,:,None]*fn).sum(1),(w[:,:,None]*fn).sum(1)],1)
            try: coef=np.linalg.solve(A,rhs)
            except:
                coef=np.zeros((len(q),3,6))
                for r in range(len(q)):
                    try: coef[r]=np.linalg.pinv(A[r])@rhs[r]
                    except: pass
            Xq=xy_q[:,0]; Yq=xy_q[:,1]
            pred=(Xq[:,None]*coef[:,0,:]+Yq[:,None]*coef[:,1,:]+coef[:,2,:]).astype(np.float32)
            pred[~vk.any(1)]=self.fa.mean(0)
            return pred,np.where(vk,dk,np.inf).min(1).astype(np.float32)

    class DenseANCCImputer:
        def __init__(self,well_ids,data_dir,spw=DENSE_SPW):
            xs,ys,anccs,wids=[],[],[],[]
            for wid in well_ids:
                p=data_dir/f'{wid}__horizontal_well.csv'
                try: df=pd.read_csv(p,usecols=['X','Y','ANCC']).dropna()
                except: continue
                if len(df)==0: continue
                ix=np.linspace(0,len(df)-1,min(spw,len(df)),dtype=int); s=df.iloc[ix]
                xs.append(s['X'].values); ys.append(s['Y'].values)
                anccs.append(s['ANCC'].values); wids.extend([wid]*len(s))
            self.xy=np.column_stack([np.concatenate(xs),np.concatenate(ys)])
            self.ancc=np.concatenate(anccs).astype(np.float32); self.wids=np.array(wids)
            self.scale=np.where(self.xy.std(0)<1e-3,1.,self.xy.std(0))
            self.tree=cKDTree(self.xy/self.scale)

        def impute(self,xy_q,self_wid=None,k=DENSE_K,nfetch=5000):
            xy_q=np.atleast_2d(xy_q); q=xy_q/self.scale; nf=min(nfetch,len(self.ancc))
            dist,idx=self.tree.query(q,k=nf,workers=-1)
            if self_wid: dist=np.where(self.wids[idx]==self_wid,np.inf,dist)
            ord=np.argpartition(dist,min(k-1,nf-1),1)[:,:k]
            dk=np.take_along_axis(dist,ord,1); ik=np.take_along_axis(idx,ord,1)
            vk=np.isfinite(dk); w=np.where(vk,1./(dk+1e-3),0.)
            sw=w.sum(1); safe=np.where(sw<1e-9,1.,sw); an=self.ancc[ik]
            ap=(an*w).sum(1)/safe; ap=np.where(sw<1e-9,float(self.ancc.mean()),ap)
            var=((an-ap[:,None])**2*w).sum(1)/safe
            return ap.astype(np.float32),np.sqrt(np.maximum(var,0.)).astype(np.float32),np.where(vk,dk,np.inf).min(1).astype(np.float32)

    hw_paths=sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
    train_wids=[p.stem.replace('__horizontal_well','') for p in hw_paths]
    FI=FormationPlaneKNN(train_wids,CFG.dataset_path / "train")
    DI=DenseANCCImputer(train_wids,CFG.dataset_path / "train")

    _FI=FI; _DI=DI
    ANCH_OFFS=np.array([-80,-40,-20,-10,-5,0,5,10,20,40,80],np.float32)
    BEAM_OFFS=np.array([-40,-20,-10,-5,-3,0,3,5,10,20,40],np.float32)
    SC_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)
    PF_OFFS  =np.array([-30,-15,-8,-4,-2,0,2,4,8,15,30],np.float32)

    def build_well(hw_path,tw_path,is_train):
        global _FI,_DI
        wid=Path(hw_path).stem.replace('__horizontal_well','')
        try:
            hw=pd.read_csv(hw_path); tw=pd.read_csv(tw_path).sort_values('TVT')
        except: return None
        if is_train and 'TVT' not in hw.columns: return None
        kn=hw[hw['TVT_input'].notna()]; ev=hw[hw['TVT_input'].isna()]
        if len(ev)==0 or len(kn)<10: return None
        if is_train and hw['TVT'].isna().all(): return None
        tw_tvt=tw['TVT'].to_numpy(np.float32); tw_gr=tw['GR'].to_numpy(np.float32)
        if len(tw_tvt)<3: return None

        pf_a,std_a=run_pf_ancc(hw,tw_tvt,tw_gr)
        if len(pf_a)==0: return None
        pf_z,std_z=run_pf_z(hw,tw_tvt,tw_gr)
        pf_use=pf_a.astype(np.float32); std_use=std_a.astype(np.float32)
        has_z=len(pf_z)==len(pf_a) and not np.any(np.isnan(pf_z))

        lk=kn.iloc[-1]; last_tvt=float(lk['TVT_input'])
        gr_full=hw['GR'].astype(float).interpolate(limit_direction='both').fillna(float(np.nanmean(tw_gr)))
        hgr=gr_full.iloc[ev.index[0]:].to_numpy(np.float32)
        kgr=gr_full.iloc[:len(kn)].to_numpy(np.float32)

        # 7 beams (Numba JIT ±2)
        bpaths={}
        for (bs,mc,es,r,tag) in BEAMS:
            bpaths[tag]=beam_search(hgr,tw_tvt,tw_gr,last_tvt,bs,mc,es,r)
        beam_ref=(bpaths['cons']+bpaths['sm5'])/2.

        # Multi-scale NCC → score-weighted ensemble
        ktvt=kn['TVT_input'].to_numpy(np.float32)
        sc_res,sc_ens=multi_scale_ncc(kgr,ktvt,hgr,hws=(8,15,25),stride=3)
        sc8,sc8s=sc_res[0]; sc15,sc15s=sc_res[1]; sc25,sc25s=sc_res[2]
        sc_cons=(sc8+sc15+sc25)/3.
        sc_trust=float(np.clip(len(kn)/200.,0.,0.6))
        hyb_ref=(1-sc_trust)*beam_ref+sc_trust*sc_ens  # use ensemble not single

        tw_at_k=np.interp(ktvt,tw_tvt,tw_gr).astype(np.float32)
        a_cal,b_cal=affine_cal(kgr,tw_at_k)
        kmd=kn['MD'].to_numpy(np.float32); kz=kn['Z'].to_numpy(np.float32)
        pfx_rmse=float(np.sqrt(np.mean((kgr-tw_at_k)**2)))
        slp_all=robust_slope(kmd,ktvt); slp_50=robust_slope(kmd[-50:],ktvt[-50:])
        slp_z=robust_slope(kz,ktvt)

        swid=wid if is_train else None
        xy_ev=ev[['X','Y']].to_numpy(np.float64); xy_kn=kn[['X','Y']].to_numpy(np.float64)
        form_ev,knn_d=_FI.impute(xy_ev,self_wid=swid)
        form_kn,_   =_FI.impute(xy_kn,self_wid=swid)
        z_kn=kn['Z'].to_numpy(np.float32); z_ev=ev['Z'].to_numpy(np.float32)

        # Per-formation: segment b_well (early/mid/late/wls) + TVT + known-zone RMSE
        tvt_fs={}; form_rmse={}; form_list=[]
        for fi2,fn in enumerate(FORMATIONS):
            b_full,b_early,b_mid,b_late,b_wls=seg_b_well(ktvt,z_kn,form_kn[:,fi2])
            tvt_f  =(-z_ev+form_ev[:,fi2]+b_full ).astype(np.float32)
            tvt_fw =(-z_ev+form_ev[:,fi2]+b_wls  ).astype(np.float32)
            tvt_f50=(-z_ev+form_ev[:,fi2]+b_late ).astype(np.float32)
            tvt_fs[f'tvtF_{fn}']=tvt_f; tvt_fs[f'tvtFw_{fn}']=tvt_fw
            tvt_fs[f'tvtF50_{fn}']=tvt_f50
            tvt_fs[f'bw_{fn}']=np.float32(b_full); tvt_fs[f'bww_{fn}']=np.float32(b_wls)
            tvt_fs[f'bw50_{fn}']=np.float32(b_late)
            tvt_fs[f'bw_early_{fn}']=np.float32(b_early)   # NEW: early segment
            tvt_fs[f'bw_mid_{fn}']=np.float32(b_mid)       # NEW: mid segment
            form_rmse[fn]=float(np.sqrt(np.mean((ktvt-(-z_kn+form_kn[:,fi2]+b_full))**2)))
            form_list.append(tvt_f)

        fs=np.stack(form_list,1)
        form_mean_d=(fs.mean(1)-last_tvt).astype(np.float32)
        form_std_d =fs.std(1).astype(np.float32)
        form_rng_d =(fs.max(1)-fs.min(1)).astype(np.float32)

        d_ancc,d_std,d_dist=_DI.impute(xy_ev,self_wid=swid)
        d_kn,d_std_kn,_=_DI.impute(xy_kn,self_wid=swid)
        b_vd=ktvt+z_kn-d_kn
        _,b_de,b_dm,b_dl,b_dw=seg_b_well(ktvt,z_kn,d_kn)
        b_d=float(np.median(b_vd))
        tvt_dense  =(-z_ev+d_ancc+b_d  ).astype(np.float32)
        tvt_densew =(-z_ev+d_ancc+b_dw ).astype(np.float32)
        tvt_dense50=(-z_ev+d_ancc+b_dl ).astype(np.float32)
        res_kn=ktvt+z_kn-d_kn
        d_rmse=float(np.sqrt(np.mean(res_kn**2))); d_bias=float(np.mean(res_kn)); d_nb_std=float(np.mean(d_std_kn))

        all_sigs=[pf_use]+[p for p in bpaths.values()]+[sc8,sc15,sc25,sc_ens,tvt_fs['tvtF_ANCC'],tvt_dense]
        sig_mat=np.stack(all_sigs,1)
        sig_std=sig_mat.std(1).astype(np.float32)
        sig_mean=(sig_mat.mean(1)-last_tvt).astype(np.float32)

        gr_s=pd.Series(gr_full.values); rolls={}
        for w in [5,21,51,101]:
            r=gr_s.rolling(w,center=True,min_periods=1)
            rolls[f'grm{w}']=r.mean().iloc[ev.index].values.astype(np.float32)
            rolls[f'grs{w}']=r.std().fillna(0).iloc[ev.index].values.astype(np.float32)
        for lag in [1,5,15,30]:
            rolls[f'glag{lag}']=gr_s.shift(lag).bfill().iloc[ev.index].values.astype(np.float32)
            rolls[f'glead{lag}']=gr_s.shift(-lag).ffill().iloc[ev.index].values.astype(np.float32)
        gr_d1=gr_s.diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
        gr_d2=gr_s.diff().diff().fillna(0.).iloc[ev.index].values.astype(np.float32)
        gr_env=gr_s.rolling(21,center=True,min_periods=1).max().iloc[ev.index].values.astype(np.float32)
        gr_nrg=np.sqrt(np.maximum((gr_s**2).rolling(21,center=True,min_periods=1).mean(),0.)
                       ).iloc[ev.index].values.astype(np.float32)

        hmd=ev['MD'].to_numpy(np.float32); md_since=hmd-float(lk['MD'])
        slp_b_all=(last_tvt+slp_all*md_since).astype(np.float32)
        slp_b_50 =(last_tvt+slp_50 *md_since).astype(np.float32)

        mdd=hw['MD'].diff().replace(0,np.nan)
        dzdmd=(hw['Z'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        dxdmd=(hw['X'].diff()/mdd).iloc[ev.index].values.astype(np.float32)
        dydmd=(hw['Y'].diff()/mdd).iloc[ev.index].values.astype(np.float32)

        nh=len(ev); frac=(np.arange(nh)/max(nh-1,1)).astype(np.float32)
        def sc(v): return np.full(nh,np.float32(v),np.float32)

        feats={
            'well':wid,'id':[f'{wid}_{i}' for i in ev.index],
            'last_known_tvt':sc(last_tvt),
            'pf_ancc':pf_use,'pf_ancc_std':std_use,
            'pf_ancc_delta':(pf_use-last_tvt).astype(np.float32),
            'pf_z':(pf_z.astype(np.float32) if has_z else sc(last_tvt)),
            'pf_z_delta':((pf_z-last_tvt).astype(np.float32) if has_z else sc(0.)),
            'pf_vs_z':((pf_use-pf_z.astype(np.float32)) if has_z else sc(0.)),
            **{f'beam_{t}_d':(p-np.float32(last_tvt)).astype(np.float32) for t,p in bpaths.items()},
            'beam_mean_d':np.stack([(p-last_tvt) for p in bpaths.values()],1).mean(1).astype(np.float32),
            'beam_std_d': np.stack([(p-last_tvt) for p in bpaths.values()],1).std(1).astype(np.float32),
            'beam_med_d': np.median(np.stack([(p-last_tvt) for p in bpaths.values()],1),1).astype(np.float32),
            'sc8_d':(sc8-np.float32(last_tvt)).astype(np.float32),'sc8_sc':sc8s,
            'sc15_d':(sc15-np.float32(last_tvt)).astype(np.float32),'sc15_sc':sc15s,
            'sc25_d':(sc25-np.float32(last_tvt)).astype(np.float32),'sc25_sc':sc25s,
            'sc_cons_d':(sc_cons-np.float32(last_tvt)).astype(np.float32),
            'sc_ens_d':(sc_ens-np.float32(last_tvt)).astype(np.float32),  # score-weighted ensemble
            'sc_trust':sc(sc_trust),'hyb_d':(hyb_ref-np.float32(last_tvt)).astype(np.float32),
            'sig_std':sig_std,'sig_mean_d':sig_mean,
            **tvt_fs,
            **{f'frm_rmse_{fn}':sc(form_rmse[fn]) for fn in FORMATIONS},
            'form_mean_d':form_mean_d,'form_std_d':form_std_d,'form_rng_d':form_rng_d,
            'spatial_ancc_d':(form_ev[:,0]-np.float32(np.interp(last_tvt,tw_tvt,tw_gr))),
            'spatial_knn_dist':knn_d,
            'dense_ancc':d_ancc,'dense_std':d_std,'dense_dist':d_dist,
            'tvt_dense_d' :(tvt_dense -last_tvt).astype(np.float32),
            'tvt_densew_d':(tvt_densew-last_tvt).astype(np.float32),
            'tvt_dense50_d':(tvt_dense50-last_tvt).astype(np.float32),
            'dense_rmse':sc(d_rmse),'dense_bias':sc(d_bias),'dense_nb_std':sc(d_nb_std),
            'pf_vs_spatial':(pf_use-tvt_fs['tvtF_ANCC']).astype(np.float32),
            'pf_vs_dense':(pf_use-tvt_dense).astype(np.float32),
            'spatial_vs_dense':(tvt_fs['tvtF_ANCC']-tvt_dense).astype(np.float32),
            'beam_vs_spatial':(bpaths['cons']-tvt_fs['tvtF_ANCC']).astype(np.float32),
            'sc_vs_beam':(sc_ens-bpaths['cons']).astype(np.float32),
            'cal_a':sc(a_cal),'cal_b':sc(b_cal),
            'pfx_rmse':sc(pfx_rmse),'known_len':sc(len(kn)),'eval_len':sc(nh),
            'slp_all':sc(slp_all),'slp_50':sc(slp_50),'slp_z':sc(slp_z),
            'slp_b_d_all':(slp_b_all-last_tvt).astype(np.float32),
            'slp_b_d_50': (slp_b_50 -last_tvt).astype(np.float32),
            'ktvt_range':sc(float(np.ptp(ktvt))),'ktvt_std':sc(float(ktvt.std())),
            'md_since':md_since,'frac':frac,'frac2':frac**2,'sqrt_frac':np.sqrt(frac),
            'z':z_ev,
            'dx':(ev['X']-float(lk['X'])).to_numpy(np.float32),
            'dy':(ev['Y']-float(lk['Y'])).to_numpy(np.float32),
            'dz':(z_ev-float(lk['Z'])).astype(np.float32),
            'dxy':np.sqrt((ev['X']-float(lk['X']))**2+(ev['Y']-float(lk['Y']))**2).to_numpy(np.float32),
            'dzdmd':dzdmd,'dxdmd':dxdmd,'dydmd':dydmd,
            'gr':hgr,'gr_d1':gr_d1,'gr_d2':gr_d2,'gr_env':gr_env,'gr_nrg':gr_nrg,
            'gr_vs_tw_anc':hgr-np.float32(np.interp(last_tvt,tw_tvt,tw_gr)),
            'gr_vs_slp_all':hgr-np.interp(slp_b_all,tw_tvt,tw_gr).astype(np.float32),
            **{f'tda{int(o)}' :hgr-np.float32(np.interp(last_tvt+o,tw_tvt,tw_gr)) for o in ANCH_OFFS},
            **{f'tdbc{int(o)}':hgr-np.interp(beam_ref+o,tw_tvt,tw_gr).astype(np.float32) for o in BEAM_OFFS},
            **{f'tdsc{int(o)}':hgr-np.interp(sc_ens+o,tw_tvt,tw_gr).astype(np.float32) for o in SC_OFFS},
            **{f'tdpf{int(o)}':hgr-np.interp(pf_use+o,tw_tvt,tw_gr).astype(np.float32) for o in PF_OFFS},
            'tw_range':sc(float(np.ptp(tw_tvt))),'tw_gr_mean':sc(float(tw_gr.mean())),
        }
        for k,v in rolls.items(): feats[k]=v
        result=pd.DataFrame(feats)
        if is_train:
            if 'TVT' not in ev.columns or ev['TVT'].isna().all(): return None
            result['target']=(ev['TVT'].to_numpy(np.float32)-np.float32(last_tvt))
        return result

    def build_dataset(paths,is_train,label):
        args=[(str(p),str(p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv'),is_train)
              for p in paths
              if (p.parent/f'{p.stem.replace("__horizontal_well","")}__typewell.csv').exists()]
        t0=time.time()
        res=Parallel(n_jobs=NCPU,prefer='threads',verbose=3)(
            delayed(build_well)(hp,tp,it) for hp,tp,it in args)
        parts=[r for r in res if r is not None]
        return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()

    if (CFG.artifacts_path / "data" / "train.csv").exists():
        train_df = pd.read_csv(CFG.artifacts_path / "data" / "train.csv", low_memory=False)
    else:
        train_paths = sorted((CFG.dataset_path / "train").glob('*__horizontal_well.csv'))
        train_df = build_dataset(train_paths, is_train=True, label="train")    

    test_paths = sorted((CFG.dataset_path / "test").glob('*__horizontal_well.csv'))
    test_df = build_dataset(test_paths, is_train=False, label="test")

    features = [c for c in train_df.columns if c not in {'well','id','target'}]

    X = train_df[features]
    y = train_df['target']
    g = train_df['well']

    X_test = test_df[features]

    lgb_params = [
        dict(
            boosting_type="gbdt", 
            num_leaves=255, 
            min_child_samples=15,
            subsample=0.8, 
            subsample_freq=1, 
            colsample_bytree=0.8,
            reg_lambda=3.0, 
            reg_alpha=0.05, 
            objective="regression",
            verbose=-1, 
            n_jobs=-1, 
            device_type="gpu", 
            gpu_use_dp=False, 
            max_bin=255,
            learning_rate=0.030, 
            n_estimators=5000, 
            seed=123
        ),
        dict(
            n_jobs=-1, 
            verbose=-1, 
            reg_alpha=10.788188919840913, 
            subsample=0.47437582748953966, 
            num_leaves=64, 
            reg_lambda=95.75401894533888, 
            n_estimators=10000,
            random_state=0,
            boosting_type='gbdt', 
            learning_rate=0.00934485794382918,
            colsample_bytree=0.39283351290380497,
            min_child_weight=0.24081152127177283, 
            min_child_samples=40,
            device='gpu',
        ),
        dict(
            n_jobs=-1, 
            verbose=-1, 
            reg_alpha=10.788188919840913, 
            subsample=0.47437582748953966, 
            num_leaves=64, 
            reg_lambda=95.75401894533888, 
            n_estimators=10000,
            random_state=29,
            boosting_type='gbdt', 
            learning_rate=0.00934485794382918,
            colsample_bytree=0.39283351290380497,
            min_child_weight=0.24081152127177283, 
            min_child_samples=40,
            device='gpu',
        ),
    ]

    cb_params = [
        dict(
            iterations=8000, 
            depth=7, 
            l2_leaf_reg=2.0,
            min_data_in_leaf=15, 
            border_count=254,
            loss_function="RMSE", 
            task_type="GPU", 
            devices="0",
            od_type="Iter", 
            od_wait=300, 
            verbose=0,
            learning_rate=0.020, 
            random_seed=7
        ),
        dict(
            iterations=8000, 
            depth=7, 
            l2_leaf_reg=2.0,
            min_data_in_leaf=15, 
            border_count=254,
            loss_function="RMSE", 
            task_type="GPU", 
            devices="0",
            od_type="Iter", 
            od_wait=300, 
            verbose=0,
            learning_rate=0.030, 
            random_seed=123
        ),
    ]

    ridge_params = {
        "random_state": 42,
        "alpha": 1.6602834637650032,
        "tol": 0.0005030247295617308,
        "positive": True,
        "fit_intercept": True
    }

    pp_params = {
        'alpha': 1.0,
        'tau': 85,
        'w_pf': 0.09
    }

    oof_preds = {}
    test_preds = {}

    overall_scores = {}
    fold_scores = {}

    for i, params in enumerate(lgb_params):   
        save_path = f"models/lightgbm-{i+1}"
    
        if (CFG.artifacts_path / save_path).exists():
            print(f"Loading lightgbm-{i+1} from disk...")
        
            trainer_path = CFG.artifacts_path / save_path / 'models.pkl'
            if not trainer_path.exists():
                trainer_paths = sorted((CFG.artifacts_path / save_path).glob('*.pkl'))
                if not trainer_paths:
                    raise FileNotFoundError(f'No pickle files found under {CFG.artifacts_path / save_path}')
                trainer_path = trainer_paths[0]
            trainer = joblib.load(trainer_path)
        
            print(f"Loaded lightgbm-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n")
        else:
     
            trainer = Trainer(
                estimator=LGBMRegressor(**params),
                task="regression",
                metric=CFG.metric,
                cv=CFG.cv,
                cv_args={"groups": g},
                use_early_stopping=True,
                verbose=True,
                save=True,
                save_path=save_path
            )
        
            trainer.fit(
                X, 
                y,
                fit_args={
                    "eval_metric": "rmse",
                    "callbacks": [
                        log_evaluation(period=250), 
                        early_stopping(stopping_rounds=250)
                    ]
                }
            )
            print("\n\n")

        oof_preds[f"lightgbm-{i+1}"] = trainer.oof_preds
        test_preds[f"lightgbm-{i+1}"] = trainer.predict(X_test)
        overall_scores[f"lightgbm-{i+1}"] = trainer.overall_score
        fold_scores[f"lightgbm-{i+1}"] = trainer.fold_scores

    for i, params in enumerate(cb_params):    
        save_path = f"models/catboost-{i+1}"
        if (CFG.artifacts_path / save_path).exists():
            print(f"Loading catboost-{i+1} from disk...")
        
            trainer_path = CFG.artifacts_path / save_path / 'models.pkl'
            if not trainer_path.exists():
                trainer_paths = sorted((CFG.artifacts_path / save_path).glob('*.pkl'))
                if not trainer_paths:
                    raise FileNotFoundError(f'No pickle files found under {CFG.artifacts_path / save_path}')
                trainer_path = trainer_paths[0]
            trainer = joblib.load(trainer_path)
        
            print(f"Loaded catboost-{i+1} with overall RMSE: {trainer.overall_score:.4f}\n")
        else:
            trainer = Trainer(
                estimator=CatBoostRegressor(**params),
                task="regression",
                metric=CFG.metric,
                cv=CFG.cv,
                cv_args={"groups": g},
                use_early_stopping=True,
                verbose=True,
                save=True,
                save_path=save_path
            )
        
            trainer.fit(
                X, 
                y,
                fit_args={
                    "verbose": 250,
                    "early_stopping_rounds": 250,
                    "use_best_model": True
                }
            )
            print("\n\n")

        oof_preds[f"catboost-{i+1}"] = trainer.oof_preds
        test_preds[f"catboost-{i+1}"] = trainer.predict(X_test)
        overall_scores[f"catboost-{i+1}"] = trainer.overall_score
        fold_scores[f"catboost-{i+1}"] = trainer.fold_scores

    oof_preds = pd.DataFrame(oof_preds)
    test_preds = pd.DataFrame(test_preds)

    ridge_trainer = Trainer(
        Ridge(**ridge_params),
        task="regression",
        metric=CFG.metric,
        cv=CFG.cv,
        cv_args={"groups": g},
        verbose=True
    )

    ridge_trainer.fit(oof_preds, y)

    ridge_oof_preds = ridge_trainer.oof_preds
    ridge_test_preds = ridge_trainer.predict(test_preds)

    overall_scores["ridge"] = ridge_trainer.overall_score
    fold_scores["ridge"] = ridge_trainer.fold_scores

    def apply_pp(df, md, pd_, alpha, tau, w_pf):
        d = md * (1-w_pf) + pd_ * w_pf
        if tau: 
            d *= (1.-np.exp(-np.maximum(df['md_since'].values,0.) / tau))
        
        return d * alpha

    def sg_smooth(df, col, sg_w=17, sg_p=3):
        df = df.copy()
    
        for _, g in df.groupby('well', sort=False):
            v = g[col].values
            n = len(v)
            wl = min(sg_w, n)
        
            if wl % 2 == 0: 
                wl -= 1
            
            if wl >= sg_p + 2: 
                v = savgol_filter(v, wl, sg_p)
            
            df.loc[g.index,col] = v
        
        return df

    base = train_df['last_known_tvt'].values
    ytrue = y.values + base

    pf_oof = (train_df['pf_ancc'].values - base)

    d = apply_pp(train_df, ridge_oof_preds, pf_oof, **pp_params)
    ridge_score = root_mean_squared_error(ytrue, base + d)

    overall_scores["ridge (pp)"] = ridge_score
    fold_scores["ridge (pp)"] = [ridge_score] * CFG.n_splits

    test_df2 = test_df.copy()
    pf_test = test_df2['pf_ancc'].values - test_df2['last_known_tvt'].values

    test_df2['pred'] = test_df2['last_known_tvt'].values + apply_pp(
        test_df2, 
        ridge_test_preds,
        pf_test, 
        **pp_params
    )
    test_df2 = sg_smooth(test_df2, 'pred')

    sample_sub = pd.read_csv(CFG.dataset_path / "sample_submission.csv")
    sub_1 = (sample_sub[['id']].merge(
        test_df2[['id', 'pred']].rename(columns={'pred':'tvt'}),
        on='id', 
        how='left'
    ))

    sub_1['tvt']=sub_1['tvt'].fillna(float(train_df['last_known_tvt'].mean()+train_df['target'].mean()))
    sub_1

    sample = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')
    sample['well']    = sample['id'].str[:8]
    sample['row_idx'] = sample['id'].str[9:].astype(int)

    train_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'train' / '*__horizontal_well.csv')))
    train_wells = [os.path.basename(f).split('__')[0] for f in train_hw_files]

    test_hw_files = sorted(glob.glob(str(CFG.dataset_path / 'test' / '*__horizontal_well.csv')))
    test_wells = [os.path.basename(f).split('__')[0] for f in test_hw_files]

    rows = []
    for i, wid in enumerate(test_wells):
        print(f'\nProcessing {i + 1}/{len(test_wells)}: {wid}...')
        hw_te, tw_te = load_well(wid, 'test')

        tvt_phys = None
        hw_tr    = None
        tw_tr    = None

        # Physical model for visible wells
        if wid in train_wells:
            try:
                hw_tr, tw_tr = load_well(wid, 'train')
                hw_te['TVT_input'] = hw_tr['TVT_input'].values
                tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
                print(f'  Physical model OK')
            except Exception as e:
                print(f'  Physical model failed: {e}')
                tvt_phys = None

        selector_code, selector_variant, selector_n_eval, selector_z_span = selector_well_code(hw_te)

        # Likelihood-weighted PF ensemble
        try:
            tw_ref = tw_tr if tw_tr is not None else tw_te
            pf_particles = int(globals().get('RIDGE_ARTIFACT_PF_N_PARTICLES', 600))
            pf_seeds = int(globals().get('RIDGE_ARTIFACT_PF_N_SEEDS', 150))
            pf_init_spread = float(globals().get('RIDGE_ARTIFACT_PF_INIT_SPREAD', 2.0))
            pf_by_scale = run_pf_lik_ensemble_scales(hw_te, tw_ref, n_particles=pf_particles, n_seeds=pf_seeds)
            tvt_pf = pf_by_scale['pf_scale_8']
            print(f'  PF lik-ensemble OK particles={pf_particles} seeds={pf_seeds} init_spread={pf_init_spread:.2f} scales={SELECTOR_SCALES}')
        except Exception as e:
            print(f'  PF failed: {e}')
            last_known = hw_te['TVT_input'].dropna()
            last_val   = float(last_known.iloc[-1]) if len(last_known) > 0 else 0.0
            tvt_pf = hw_te['TVT_input'].fillna(last_val).values.astype(float)
            pf_by_scale = {f'pf_scale_{scale:g}': tvt_pf.copy() for scale in SELECTOR_SCALES}

        # Beam search ensemble
        try:
            tw_ref = tw_tr if tw_tr is not None else tw_te
            tvt_beam = run_beam_ensemble(hw_te, tw_ref)
            print(f'  Beam 14-config ensemble OK')
        except Exception as e:
            print(f'  Beam failed: {e}')
            tvt_beam = tvt_pf.copy()

        # Selector blending
        last_known = hw_te['TVT_input'].dropna()
        last_known_tvt = float(last_known.iloc[-1]) if len(last_known) > 0 else float(np.nanmean(tvt_pf))
        tvt_selector = apply_selector_variant(selector_variant, pf_by_scale, tvt_beam, last_known_tvt)
        print(
            f'  Selector code={selector_code} variant={selector_variant} '
            f'n_eval={selector_n_eval:.0f} z_span={selector_z_span:.3f}'
        )

        ws = sample[sample['well'] == wid]
        for _, row in ws.iterrows():
            ridx = int(row['row_idx'])
            if tvt_phys is not None:
                tvt_val = float(tvt_phys.iloc[ridx])
            else:
                tvt_val = float(tvt_selector[ridx])
            rows.append({'id': row['id'], 'tvt': tvt_val})
        print(f'  Added {len(ws)} rows')

    sub_2 = pd.DataFrame(rows)


    # Final artifact-ridge blend. w_r is configured in the visible settings cell.
    configured_ridge_weight = float(np.clip(float(RIDGE_ARTIFACT_RIDGE_WEIGHT), 0.0, 1.0))
    merged = sub_1.merge(sub_2, on='id', suffixes=('_ridge', '_heur'))
    sample_order = pd.read_csv(CFG.dataset_path / 'sample_submission.csv')[['id']]

    def _align_to_sample(frame, label):
        frame = frame[['id', 'tvt']].copy()
        frame['_id_key'] = frame['id'].astype(str)
        sample_keys = sample_order.copy()
        sample_keys['_id_key'] = sample_keys['id'].astype(str)
        if frame['_id_key'].duplicated().any():
            dup = frame.loc[frame['_id_key'].duplicated(), 'id'].head(10).tolist()
            raise RuntimeError(f'{label}: duplicated ids: {dup}')
        missing = sorted(set(sample_keys['_id_key']) - set(frame['_id_key']))
        extra = sorted(set(frame['_id_key']) - set(sample_keys['_id_key']))
        if missing or extra:
            raise RuntimeError(f'{label}: id mismatch missing={len(missing)} extra={len(extra)} examples={missing[:5] or extra[:5]}')
        aligned = sample_keys.merge(frame[['_id_key', 'tvt']], on='_id_key', how='left')
        aligned = pd.DataFrame({'id': sample_order['id'].to_numpy(), 'tvt': aligned['tvt'].to_numpy()})
        aligned['tvt'] = pd.to_numeric(aligned['tvt'], errors='coerce')
        if aligned['tvt'].isna().any():
            bad = aligned.loc[aligned['tvt'].isna(), 'id'].head(10).tolist()
            raise RuntimeError(f'{label}: NaN after sample alignment: {bad}')
        if not np.isfinite(aligned['tvt'].to_numpy(dtype=float)).all():
            raise RuntimeError(f'{label}: non-finite TVT values')
        return aligned[['id', 'tvt']]

    def _make_ridge_heuristic_blend(weight, label):
        w = float(np.clip(float(weight), 0.0, 1.0))
        cand = merged.assign(
            tvt=lambda x, ww=w: ww * x['tvt_ridge'] + (1.0 - ww) * x['tvt_heur']
        )[['id', 'tvt']]
        return _align_to_sample(cand, label)

    out_dir = Path(globals().get('OUTPUT_DIR', Path('/kaggle/working')))
    out_dir.mkdir(parents=True, exist_ok=True)
    final_output = Path(globals().get('FINAL_SUBMISSION_OUTPUT', Path('submission.csv')))

    profile_name = str(globals().get('RIDGE_ARTIFACT_PROFILE_LABEL', 'ridge_artifact_experiment'))
    def _robust_polyfit_predict(s, y, degree=5, robust_iters=4, robust_c=2.0):
        s = np.asarray(s, dtype=float)
        y = np.asarray(y, dtype=float)
        mask = np.isfinite(s) & np.isfinite(y)
        if mask.sum() < int(degree) + 2:
            return y.copy()
        degree = int(min(int(degree), max(1, mask.sum() - 2)))
        coef = np.polyfit(s[mask], y[mask], degree)
        for _ in range(int(robust_iters)):
            residual = y[mask] - np.polyval(coef, s[mask])
            scale = np.median(np.abs(residual)) * 1.4826 + 1e-6
            weights = 1.0 / (1.0 + (residual / (float(robust_c) * scale)) ** 2)
            coef = np.polyfit(s[mask], y[mask], degree, w=weights)
        pred = np.asarray(np.polyval(coef, s), dtype=float)
        pred[~np.isfinite(pred)] = y[~np.isfinite(pred)]
        return pred

    def _project_submission_by_well(frame, degree=5, robust_iters=4, robust_c=2.0):
        projected = _align_to_sample(frame, 'projection_input').copy()
        projected['well'] = projected['id'].astype(str).str[:8]
        projected['row_idx'] = projected['id'].astype(str).str[9:].astype(int)
        out_values = dict(zip(projected['id'].astype(str), projected['tvt'].astype(float)))
        summary_rows = []
        n_projected = 0
        for wid, group in projected.groupby('well', sort=False):
            try:
                hw = pd.read_csv(CFG.dataset_path / 'test' / f'{wid}__horizontal_well.csv')
                known = hw[hw['TVT_input'].notna()]
                if len(known) < 5:
                    summary_rows.append({'well': wid, 'projected': False, 'reason': 'too_few_known_rows'})
                    continue
                last = known.iloc[-1]
                anchor = float(last['TVT_input']) + float(last['Z'])
                start_md = float(last['MD'])
                end_md = float(hw['MD'].iloc[-1])
                ordered = group.sort_values('row_idx')
                row_idx = ordered['row_idx'].to_numpy(dtype=int)
                z = hw['Z'].to_numpy(dtype=float)[row_idx]
                md = hw['MD'].to_numpy(dtype=float)[row_idx]
                s = (md - start_md) / max(end_md - start_md, 1e-6)
                tvt = ordered['tvt'].to_numpy(dtype=float)
                u = tvt + z - anchor
                u_fit = _robust_polyfit_predict(s, u, degree=degree, robust_iters=robust_iters, robust_c=robust_c)
                tvt_fit = anchor + u_fit - z
                if not np.all(np.isfinite(tvt_fit)):
                    summary_rows.append({'well': wid, 'projected': False, 'reason': 'non_finite_projection'})
                    continue
                diff = tvt_fit - tvt
                for rid, value in zip(ordered['id'].astype(str), tvt_fit):
                    out_values[rid] = float(value)
                n_projected += 1
                summary_rows.append({
                    'well': wid,
                    'projected': True,
                    'reason': 'ok',
                    'rows': int(len(ordered)),
                    'mean_abs_adjustment': float(np.mean(np.abs(diff))),
                    'max_abs_adjustment': float(np.max(np.abs(diff))),
                })
            except Exception as exc:
                summary_rows.append({'well': wid, 'projected': False, 'reason': str(exc)[:160]})
        final = sample_order.copy()
        final['tvt'] = final['id'].astype(str).map(out_values).astype(float)
        final = _align_to_sample(final, 'projection_output')
        return final, pd.DataFrame(summary_rows), n_projected

    selected_ridge_weight = configured_ridge_weight
    selected_raw = _make_ridge_heuristic_blend(selected_ridge_weight, profile_name)
    raw_profile_file = out_dir / f'submission_{profile_name}_raw.csv'
    selected_raw.to_csv(raw_profile_file, index=False)

    projection_enabled = bool(globals().get('RIDGE_ARTIFACT_APPLY_PROJECTION', False))
    projection_degree = int(globals().get('RIDGE_ARTIFACT_PROJECTION_DEGREE', 5))
    projection_iters = int(globals().get('RIDGE_ARTIFACT_PROJECTION_ROBUST_ITERS', 4))
    projection_c = float(globals().get('RIDGE_ARTIFACT_PROJECTION_ROBUST_C', 2.0))
    if projection_enabled:
        selected, projection_summary, projected_wells = _project_submission_by_well(
            selected_raw,
            degree=projection_degree,
            robust_iters=projection_iters,
            robust_c=projection_c,
        )
        projection_summary.to_csv(out_dir / f'ridge_artifact_projection_summary_{profile_name}.csv', index=False)
        profile_base_file = out_dir / f'submission_{profile_name}_projected.csv'
        selected.to_csv(profile_base_file, index=False)
    else:
        selected = selected_raw
        projected_wells = 0
        projection_summary = pd.DataFrame()
        profile_base_file = out_dir / f'submission_{profile_name}.csv'
        selected.to_csv(profile_base_file, index=False)

    candidate_rows = [{
        'file': profile_base_file.name,
        'raw_file': raw_profile_file.name,
        'candidate_type': 'ridge_parameter_experiment' if bool(globals().get('RUN_RIDGE_ARTIFACT_EXPERIMENT', False)) else 'ridge_reference_exact',
        'ridge_weight_w_r': selected_ridge_weight,
        'heuristic_weight_1_minus_w_r': 1.0 - selected_ridge_weight,
        'pf_init_spread_sigma_0': float(globals().get('RIDGE_ARTIFACT_PF_INIT_SPREAD', 2.0)),
        'pf_particles_N_p': int(globals().get('RIDGE_ARTIFACT_PF_N_PARTICLES', 600)),
        'pf_seeds_S': int(globals().get('RIDGE_ARTIFACT_PF_N_SEEDS', 150)),
        'projection_enabled': projection_enabled,
        'projection_degree_d': projection_degree if projection_enabled else None,
        'projection_robust_iters': projection_iters if projection_enabled else None,
        'projected_wells': int(projected_wells),
        'selected_for_submission_csv': True,
        'rows': int(len(selected)),
        'tvt_mean': float(selected['tvt'].mean()),
        'tvt_std': float(selected['tvt'].std()),
        'tvt_min': float(selected['tvt'].min()),
        'tvt_max': float(selected['tvt'].max()),
    }]

    profile_out = out_dir / f'submission_{profile_name}.csv'
    selected.to_csv(final_output, index=False)
    selected.to_csv(profile_out, index=False)

    pd.DataFrame(candidate_rows).to_csv(out_dir / 'ridge_artifact_candidate_report.csv', index=False)
    pd.DataFrame([{
        'profile': profile_name,
        'artifact_root': str(CFG.artifacts_path),
        'ridge_weight_w_r': selected_ridge_weight,
        'heuristic_weight_1_minus_w_r': 1.0 - selected_ridge_weight,
        'ridge_pf_particles_N_p': int(globals().get('RIDGE_ARTIFACT_PF_N_PARTICLES', 600)),
        'ridge_pf_seeds_S': int(globals().get('RIDGE_ARTIFACT_PF_N_SEEDS', 150)),
        'ridge_pf_init_spread_sigma_0': float(globals().get('RIDGE_ARTIFACT_PF_INIT_SPREAD', 2.0)),
        'output_file': str(final_output),
        'profile_output_file': profile_out.name,
        'raw_profile_output_file': raw_profile_file.name,
        'projection_enabled': projection_enabled,
        'projection_degree_d': projection_degree if projection_enabled else None,
        'projection_robust_iters': projection_iters if projection_enabled else None,
        'projected_wells': int(projected_wells),
        'rows': int(len(selected)),
        'tvt_mean': float(selected['tvt'].mean()),
        'tvt_std': float(selected['tvt'].std()),
        'tvt_min': float(selected['tvt'].min()),
        'tvt_max': float(selected['tvt'].max()),
    }]).to_csv(out_dir / 'ridge_artifact_summary.csv', index=False)

    globals()['FINAL_SELECTED_BASE_SOURCE'] = final_output
    globals()['FINAL_BASE_SOURCE_LABEL'] = profile_name
    pf_particles_used = int(globals().get('RIDGE_ARTIFACT_PF_N_PARTICLES', 600))
    pf_seeds_used = int(globals().get('RIDGE_ARTIFACT_PF_N_SEEDS', 150))
    pf_init_spread_used = float(globals().get('RIDGE_ARTIFACT_PF_INIT_SPREAD', 2.0))
    print(
        f'Saved {final_output} using {profile_name}: '
        f'w_r={selected_ridge_weight:.3f}, heuristic={1.0 - selected_ridge_weight:.3f}, '
        f'pf_particles={pf_particles_used}, pf_seeds={pf_seeds_used}, init_spread={pf_init_spread_used:.2f}, '
        f'projection={projection_enabled}, degree={projection_degree if projection_enabled else None}'
    )
    try:
        display(selected.head())
    except Exception:
        print(selected.head())


[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done   3 out of   3 | elapsed:    7.5s finished


Loading lightgbm-1 from disk...
Loaded lightgbm-1 with overall RMSE: 10.7668

Loading lightgbm-2 from disk...
Loaded lightgbm-2 with overall RMSE: 10.4852

Loading lightgbm-3 from disk...
Loaded lightgbm-3 with overall RMSE: 10.4733

Loading catboost-1 from disk...
Loaded catboost-1 with overall RMSE: 10.5750

Loading catboost-2 from disk...
Loaded catboost-2 with overall RMSE: 10.5550

Training Ridge

--- Fold 0 - root_mean_squared_error: 9.4973 - Time: 0.58 s
--- Fold 1 - root_mean_squared_error: 10.8118 - Time: 0.59 s
--- Fold 2 - root_mean_squared_error: 9.3836 - Time: 0.57 s
--- Fold 3 - root_mean_squared_error: 11.4277 - Time: 0.57 s
--- Fold 4 - root_mean_squared_error: 10.8220 - Time: 0.63 s

------ Overall root_mean_squared_error: 10.4197 - Mean root_mean_squared_error: 10.3885 ± 0.8064 - Time: 6.01 s

Processing 1/3: 000d7d20...
  Physical model OK
  PF lik-ensemble OK particles=500 seeds=128 init_spread=4.50 scales=(3.0, 5.0, 8.0, 12.0)
  Beam 14-config ensemble OK
  Selecto

,id,tvt
0,000d7d20_1442,11758
1,000d7d20_1443,11758
2,000d7d20_1444,11758
3,000d7d20_1445,11758
4,000d7d20_1446,11758


In [7]:
# Optional external correction profiles are not active in the current profile set.
sidecar_submission = None


In [8]:
# Preserve the selected profile submission unchanged.
if FINAL_SUBMISSION_OUTPUT.exists():
    FINAL_SIDECAR_SOURCE_LABEL = str(globals().get('FINAL_BASE_SOURCE_LABEL', 'base_only'))
    FINAL_SIDECAR_AVAILABLE = False
    FINAL_SIDECAR_AUTO_DISABLED_REASON = ''


## Final Submission Contract Guard

Final sanity check before writing `submission.csv`:

| Check | Requirement |
|---|---|
| columns | `id,tvt` only |
| rows | same count as `sample_submission.csv` |
| ids | same order as sample |
| tvt | numeric, finite, non-missing |
| source | final TVT trajectory after optional conservative correction |

If any condition fails, the notebook raises an error instead of silently writing a bad file.


In [9]:
# Final v7 submission contract guard.
FINAL_V7_SOURCE = Path(globals().get('FINAL_SELECTED_BASE_SOURCE', globals().get('SUPER_STACK_SUBMISSION_OUTPUT', FINAL_SUBMISSION_OUTPUT)))
if bool(globals().get('RUN_SUPER_STACK_SOLUTION', False)) and not FINAL_V7_SOURCE.exists():
    raise RuntimeError(f'Expected super-stack submission was not produced: {FINAL_V7_SOURCE}')

if FINAL_SUBMISSION_OUTPUT.exists() and SAMPLE_SUBMISSION.exists():
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    final = pd.read_csv(FINAL_SUBMISSION_OUTPUT)
    if list(final.columns) != ['id', 'tvt']:
        raise RuntimeError(f'Final submission columns must be [id, tvt], got {list(final.columns)}')
    if len(final) != len(sample):
        raise RuntimeError(f'Final submission row mismatch: got {len(final)}, expected {len(sample)}')
    if not final['id'].equals(sample['id']):
        raise RuntimeError('Final submission ids do not match sample_submission order.')
    final['tvt'] = pd.to_numeric(final['tvt'], errors='coerce')
    if final['tvt'].isna().any() or not np.isfinite(final['tvt'].to_numpy(dtype=float)).all():
        raise RuntimeError('Final submission contains missing or non-finite tvt values.')
    final[['id', 'tvt']].to_csv(FINAL_SUBMISSION_OUTPUT, index=False)
    contract_summary = pd.DataFrame([{
        'final_submission': str(FINAL_SUBMISSION_OUTPUT),
        'submission_profile': str(globals().get('SUBMISSION_PROFILE', 'unknown')),
        'source_label': str(globals().get('FINAL_BASE_SOURCE_LABEL', globals().get('FINAL_SIDECAR_SOURCE_LABEL', 'base_only'))),
        'source_file': str(FINAL_V7_SOURCE) if FINAL_V7_SOURCE.exists() else str(FINAL_SUBMISSION_OUTPUT),
        'rows': int(len(final)),
        'columns': ','.join(final.columns),
        'tvt_mean': float(final['tvt'].mean()),
        'tvt_std': float(final['tvt'].std()),
        'tvt_min': float(final['tvt'].min()),
        'tvt_max': float(final['tvt'].max()),
        'contract_pass': True,
    }])
    contract_summary.to_csv(OUTPUT_DIR / 'submission_contract_guard_summary_v7_final.csv', index=False)
    display(contract_summary)
else:
    print('Final submission guard skipped because submission.csv or sample_submission.csv is unavailable.')


,final_submission,submission_profile,source_label,source_file,rows,columns,tvt_mean,tvt_std,tvt_min,tvt_max,contract_pass
0,/kaggle/working/submission.csv,ridge_artifact_experiment,ridge_exp_w030_p500_s128_sp45_proj_d4,/kaggle/working/submission.csv,14151,"id,tvt",11904,278.5,11590,12240,True


## References

This notebook uses a compact set of ideas from public ROGII notebooks and related baselines:

- PF selector / physical model: https://www.kaggle.com/code/aiwody/physical-model-less-overfitting-noise
- PF selector rerun: https://www.kaggle.com/code/aidensong123/rogii-sel15-rerun
- Ridge artifact reference: https://www.kaggle.com/code/overvalueawareness/wellbore-geology-prediction-ridge/notebook
- Ridge artifact reference: https://www.kaggle.com/code/ravaghi/wellbore-geology-prediction-ridge
- Ridge SP45 variant: https://www.kaggle.com/code/needless090/rogii-ridge-sp45
- Better solution LB 9.956: https://www.kaggle.com/code/romantamrazov/rogii-better-solution-lb-9-956
- Super solution LB top 3: https://www.kaggle.com/code/romantamrazov/rogii-super-solution-lb-top-3
- Physics-informed baseline: https://www.kaggle.com/code/karnakbaevarthur/physics-informed-baseline?scriptVersionId=317950936
- Triple-signal beam search / dual PF / LightGBM: https://www.kaggle.com/code/shinyanagai123/triple-signal-beam-search-dual-pf-lightgbm
- Plane-fit formation-top KNN: https://www.kaggle.com/code/konbu17/rogii-plane-fit-formation-top-knn
- Wellbore geology prediction baseline: https://www.kaggle.com/code/vishwasmishra1234/rogii-wellbore-geology-prediction
- XGB starter CV 15: https://www.kaggle.com/code/cdeotte/xgb-starter-cv-15
